# Thesis Evaluation Notebook

This notebook is the dissertation-facing benchmark analysis entry point for scenario validation, static benchmarking, dynamic comparisons, hybrid analysis, statistical testing, and export generation.


## 0. Notebook setup


In [1]:
from __future__ import annotations

import json
import math
import re
import warnings
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    from scipy import stats
except Exception:
    stats = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK_ROOT = Path.cwd().resolve()
if not (NOTEBOOK_ROOT / "load_results.py").exists():
    if (NOTEBOOK_ROOT / "evaluation" / "load_results.py").exists():
        NOTEBOOK_ROOT = NOTEBOOK_ROOT / "evaluation"
    else:
        raise RuntimeError("Run this notebook from the evaluation directory or repo root.")

REPO_ROOT = NOTEBOOK_ROOT.parent
SCENARIO_ROOT = REPO_ROOT / "data" / "instances" / "generated"
RESULTS_ROOT = REPO_ROOT / "data" / "results"
DYNAMIC_RESULT_ROOTS = {
    "heuristic": [
        RESULTS_ROOT / "heuristic_fisso",
        RESULTS_ROOT / "heuristic",
    ],
    "ai": [
        RESULTS_ROOT / "ai",
    ],
    "hybrid": [
        RESULTS_ROOT / "hybrid",
        RESULTS_ROOT / "hybrid_placeholder",
        RESULTS_ROOT / "hybrid-results",
    ],
}
STATIC_RESULTS_ROOT = RESULTS_ROOT / "static"
ORACLE_RESULTS_ROOT = RESULTS_ROOT / "oracle"
OUTPUT_ROOT = NOTEBOOK_ROOT / "outputs" / "thesis_evaluation"
TABLES_ROOT = OUTPUT_ROOT / "tables"
FIGURES_ROOT = OUTPUT_ROOT / "figures"

EXPECTED_INSTANCE_COUNT = 56
EXPECTED_SCENARIO_COUNT = 3024
EXPECTED_EVALUATION_SIZES = (50, 75)
EXPECTED_DODS = (0.3, 0.5, 0.7)
EXPECTED_CUTOFFS = (0.2, 0.5, 0.8)
EXPECTED_SEEDS = (851657, 3765530, 9547451)
EXPECTED_SCENARIOS_PER_SIZE_DOD_CUTOFF = 56 * 3
EXPECTED_AI_HYBRID_FAMILIES = [
    ("general", 50),
    ("general", 75),
    ("lateness", 50),
    ("lateness", 75),
    ("solomon", 50),
    ("solomon", 75),
    ("dynamic", 50),
    ("dynamic", 75),
]
EXPECTED_STRATEGIES = ["greedy", "sampling", "multistart"]
EXPECTED_DYNAMIC_CONFIGS_PER_MODALITY = len(EXPECTED_AI_HYBRID_FAMILIES) * len(EXPECTED_STRATEGIES)

for folder in [
    OUTPUT_ROOT,
    TABLES_ROOT,
    FIGURES_ROOT / "scenario_validation",
    FIGURES_ROOT / "static_oracle",
    FIGURES_ROOT / "dynamic_comparison",
    FIGURES_ROOT / "hybrid_vs_ai",
    FIGURES_ROOT / "strategy",
    FIGURES_ROOT / "model_family",
    FIGURES_ROOT / "sensitivity",
    FIGURES_ROOT / "solomon_structure",
    FIGURES_ROOT / "static_vs_dynamic",
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Scenario root: {SCENARIO_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")


Scenario root: /Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/data/instances/generated
Results root: /Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/data/results
Output root: /Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation


In [2]:
SCENARIO_COLUMNS = [
    "scenario_id",
    "instance_id",
    "evaluation_size",
    "degree_of_dynamicity",
    "cutoff_time",
    "seed",
    "dynamic_customer_ids",
    "reveal_times",
    "feasible",
    "dropped_reason",
    "solomon_class",
    "solomon_type",
    "num_dynamic_customers",
    "num_static_customers",
    "min_reveal_time",
    "mean_reveal_time",
    "max_reveal_time",
    "reveal_time_std",
    "mean_reveal_fraction",
    "max_reveal_fraction",
    "source_file",
]

DYNAMIC_RESULT_COLUMNS = [
    "run_id",
    "scenario_id",
    "instance_id",
    "evaluation_size",
    "degree_of_dynamicity",
    "cutoff_time",
    "seed",
    "model_type",
    "model_name",
    "strategy",
    "model_family",
    "model_customer_size",
    "allow_rejection",
    "feasible",
    "error_message",
    "num_vehicles",
    "total_distance",
    "total_cost",
    "customers_served",
    "customers_rejected",
    "service_level",
    "rejection_rate",
    "average_lateness",
    "computational_time",
    "routes",
    "timestamp",
    "solomon_class",
    "solomon_type",
    "lateness_proxy",
    "cost_per_customer",
    "distance_per_customer",
    "customers_per_vehicle",
    "method_label",
    "configuration_id",
    "num_reoptimization_events",
    "runtime_per_customer",
    "runtime_per_event",
    "total_lateness",
    "lateness_count",
    "max_lateness",
    "mean_lateness",
    "source_file",
]

STATIC_RESULT_COLUMNS = [
    "static_instance_id",
    "instance_id",
    "evaluation_size",
    "model_type",
    "model_name",
    "strategy",
    "model_family",
    "model_customer_size",
    "feasible",
    "num_vehicles",
    "total_distance",
    "total_cost",
    "customers_served",
    "customers_rejected",
    "service_level",
    "rejection_rate",
    "average_lateness",
    "computational_time",
    "routes",
    "timestamp",
    "solomon_class",
    "solomon_type",
    "method_label",
    "configuration_id",
    "cost_per_customer",
    "distance_per_customer",
    "customers_per_vehicle",
    "source_file",
]

ORACLE_RESULT_COLUMNS = [
    "instance_id",
    "evaluation_size",
    "feasible",
    "num_vehicles",
    "total_distance",
    "total_cost",
    "computational_time",
    "solomon_class",
    "solomon_type",
    "source_file",
]


def empty_df(columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame(columns=columns)


def ensure_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column not in df.columns:
            df[column] = pd.NA
    return df[columns + [column for column in df.columns if column not in columns]]


def first_existing(paths: list[Path]) -> list[Path]:
    return [path for path in paths if path.exists()]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def format_token(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "na"
    text = str(value)
    return text.replace(".", "p")


def safe_divide(numerator, denominator, default=np.nan):
    try:
        if denominator in (0, None) or pd.isna(denominator):
            return default
        if numerator is None or pd.isna(numerator):
            return default
        return numerator / denominator
    except Exception:
        return default


def instance_stem(instance_id: str | None) -> str | None:
    if not instance_id or pd.isna(instance_id):
        return None
    return Path(str(instance_id)).stem.upper()


def parse_solomon_metadata(instance_id: str | None) -> tuple[str | None, int | None]:
    stem = instance_stem(instance_id)
    if not stem:
        return None, None
    match = re.match(r"^(RC|R|C)(\d)", stem)
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


def build_scenario_id(instance_id, evaluation_size, seed, degree_of_dynamicity, cutoff_time) -> str | None:
    stem = instance_stem(instance_id)
    if not stem:
        return None
    return f"{stem}_n{int(evaluation_size)}_seed{int(seed)}_dod{format_token(degree_of_dynamicity)}_cut{format_token(cutoff_time)}"


def build_static_instance_id(instance_id, evaluation_size) -> str | None:
    stem = instance_stem(instance_id)
    if not stem or pd.isna(evaluation_size):
        return None
    return f"{stem}_n{int(evaluation_size)}"


def seed_from_text(text: str | None):
    if not text:
        return None
    match = re.search(r"seed(\d+)", str(text))
    return int(match.group(1)) if match else None


def parse_filename_tokens(path: Path) -> dict:
    stem = path.stem
    parts = stem.split("__")
    parsed = {
        "instance_id": None,
        "evaluation_size": None,
        "model_name": None,
        "seed": None,
        "degree_of_dynamicity": None,
        "cutoff_time": None,
        "strategy": None,
        "num_samples": None,
        "num_starts": None,
        "num_augment": None,
    }
    if parts:
        parsed["instance_id"] = f"{parts[0]}.txt"
    if len(parts) > 1 and parts[1].startswith("n"):
        try:
            parsed["evaluation_size"] = int(parts[1][1:])
        except Exception:
            pass
    if len(parts) > 2:
        parsed["model_name"] = parts[2]
    for token in parts[3:]:
        if token.startswith("seed"):
            parsed["seed"] = int(token.replace("seed", ""))
        elif token.startswith("dod"):
            parsed["degree_of_dynamicity"] = float(token.replace("dod", "").replace("p", "."))
        elif token.startswith("cut"):
            parsed["cutoff_time"] = float(token.replace("cut", "").replace("p", "."))
        elif token.startswith("decode-"):
            parsed["strategy"] = token.replace("decode-", "")
        elif token.startswith("samples-"):
            value = token.replace("samples-", "")
            parsed["num_samples"] = None if value == "na" else int(value)
        elif token.startswith("starts-"):
            value = token.replace("starts-", "")
            parsed["num_starts"] = None if value == "na" else int(value)
        elif token.startswith("augment-"):
            value = token.replace("augment-", "")
            parsed["num_augment"] = None if value == "na" else int(value)
    return parsed


def parse_scenario_filename(path: Path) -> dict:
    stem = path.stem
    match = re.match(r"^(?P<instance>[A-Za-z0-9]+)_n(?P<size>\d+)_seed(?P<seed>\d+)_dod(?P<dod>[0-9p]+)_cut(?P<cut>[0-9p]+)$", stem)
    if not match:
        return {
            "instance_id": None,
            "evaluation_size": None,
            "seed": None,
            "degree_of_dynamicity": None,
            "cutoff_time": None,
        }
    return {
        "instance_id": f"{match.group('instance')}.txt",
        "evaluation_size": int(match.group("size")),
        "seed": int(match.group("seed")),
        "degree_of_dynamicity": float(match.group("dod").replace("p", ".")),
        "cutoff_time": float(match.group("cut").replace("p", ".")),
    }


def parse_strategy(value, path: Path | None = None) -> str:
    text = str(value or "")
    if text and text.lower() in {"greedy", "sampling", "multistart"}:
        return text.lower()
    for candidate in [text, path.stem if path else ""]:
        lowered = candidate.lower()
        if "multistart" in lowered:
            return "multistart"
        if "sampling" in lowered:
            return "sampling"
        if "greedy" in lowered:
            return "greedy"
    return "NA"


def parse_model_family_and_size(model_name: str | None) -> tuple[str, str]:
    name = str(model_name or "").lower()
    if name in {"", "nan", "none"}:
        return "NA", "NA"
    if "with_lateness" in name:
        family = "lateness"
    elif "solomon" in name:
        family = "solomon"
    elif "general" in name:
        family = "general"
    elif "dynamic" in name:
        family = "dynamic"
    elif "ortools" in name or "heuristic" in name or "oracle" in name:
        return "NA", "NA"
    else:
        family = "NA"
    match = re.search(r"(50|75)", name)
    size = match.group(1) if match else "NA"
    return family, size


def method_label(model_type, strategy, model_family, model_customer_size, model_name) -> str:
    parts = [str(model_type).upper()]
    if strategy not in {None, "NA", pd.NA}:
        parts.append(str(strategy))
    if model_family not in {None, "NA", pd.NA}:
        parts.append(f"{model_family}-{model_customer_size}")
    elif model_name not in {None, "", "nan"}:
        parts.append(str(model_name))
    return " | ".join(parts)


def configuration_id(model_type, strategy, model_family, model_customer_size, model_name, allow_rejection) -> str:
    return "|".join(
        [
            str(model_type or "NA"),
            str(strategy or "NA"),
            str(model_family or "NA"),
            str(model_customer_size or "NA"),
            str(model_name or "NA"),
            f"reject-{int(bool(allow_rejection))}",
        ]
    )


def placeholder_figure(title: str, message: str, figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    ax.text(0.5, 0.6, title, ha="center", va="center", fontsize=14, fontweight="bold")
    ax.text(0.5, 0.4, message, ha="center", va="center", fontsize=10, wrap=True)
    return fig, ax


def save_figure(fig, folder_name: str, file_name: str):
    path = FIGURES_ROOT / folder_name / file_name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path


def save_table(df: pd.DataFrame, file_name: str):
    path = TABLES_ROOT / file_name
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def print_table(label: str, df: pd.DataFrame, rows: int = 10):
    print(f"{label}: {len(df)} rows x {len(df.columns)} columns")
    display(df.head(rows))


def maybe_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def maybe_datetime(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


def scenario_reveal_times(payload: dict) -> list[float]:
    direct = payload.get("reveal_times")
    if isinstance(direct, dict):
        return [float(value) for _, value in sorted(direct.items(), key=lambda item: int(item[0]))]
    if isinstance(direct, list):
        return [float(value) for value in direct]
    dynamic_ids = set(payload.get("dynamic_customer_ids") or [])
    customers = payload.get("instance", {}).get("customers", [])
    reveal = []
    for customer in customers:
        if customer.get("id") in dynamic_ids:
            ready_time = customer.get("ready_time")
            if ready_time is not None:
                reveal.append(float(ready_time))
    return reveal


def load_scenarios_df() -> pd.DataFrame:
    rows = []
    for path in sorted(SCENARIO_ROOT.glob("*.json")):
        payload = read_json(path)
        meta = parse_scenario_filename(path)
        instance_id = payload.get("instance_id") or meta["instance_id"]
        evaluation_size = payload.get("evaluation_size") or meta["evaluation_size"]
        degree_of_dynamicity = payload.get("degree_of_dynamicity") or meta["degree_of_dynamicity"]
        cutoff_time = payload.get("cutoff_time") or meta["cutoff_time"]
        seed = payload.get("seed") or meta["seed"] or seed_from_text(path.name)
        reveal_times = scenario_reveal_times(payload)
        dynamic_customer_ids = payload.get("dynamic_customer_ids") or []
        solomon_class, solomon_type = parse_solomon_metadata(instance_id)
        customers = payload.get("instance", {}).get("customers", [])
        horizon = max([customer.get("due_time", 0) for customer in customers] + [1])
        reveal_fractions = [safe_divide(value, horizon, np.nan) for value in reveal_times]
        rows.append(
            {
                "scenario_id": path.stem,
                "instance_id": instance_id,
                "evaluation_size": evaluation_size,
                "degree_of_dynamicity": degree_of_dynamicity,
                "cutoff_time": cutoff_time,
                "seed": seed,
                "dynamic_customer_ids": dynamic_customer_ids,
                "reveal_times": reveal_times,
                "feasible": payload.get("feasible"),
                "dropped_reason": payload.get("dropped_reason"),
                "solomon_class": solomon_class,
                "solomon_type": solomon_type,
                "num_dynamic_customers": len(dynamic_customer_ids),
                "num_static_customers": safe_divide(
                    evaluation_size - len(dynamic_customer_ids) if evaluation_size is not None else np.nan,
                    1,
                    np.nan,
                ),
                "min_reveal_time": np.min(reveal_times) if reveal_times else np.nan,
                "mean_reveal_time": np.mean(reveal_times) if reveal_times else np.nan,
                "max_reveal_time": np.max(reveal_times) if reveal_times else np.nan,
                "reveal_time_std": np.std(reveal_times, ddof=0) if reveal_times else np.nan,
                "mean_reveal_fraction": np.mean(reveal_fractions) if reveal_fractions else np.nan,
                "max_reveal_fraction": np.max(reveal_fractions) if reveal_fractions else np.nan,
                "source_file": str(path.resolve()),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return empty_df(SCENARIO_COLUMNS)
    df = maybe_numeric(df, ["evaluation_size", "degree_of_dynamicity", "cutoff_time", "seed", "num_dynamic_customers", "num_static_customers", "min_reveal_time", "mean_reveal_time", "max_reveal_time", "reveal_time_std", "mean_reveal_fraction", "max_reveal_fraction"])
    return ensure_columns(df, SCENARIO_COLUMNS)


def discover_result_files(modality: str) -> list[Path]:
    result_files = []
    for root in DYNAMIC_RESULT_ROOTS.get(modality, []):
        if root.exists():
            result_files.extend(sorted(root.glob("*.json")))
    return result_files


def normalize_result_payload(payload: dict, path: Path, default_model_type: str) -> dict:
    meta = parse_filename_tokens(path)
    instance_id = payload.get("instance_id") or meta["instance_id"]
    evaluation_size = payload.get("evaluation_size") or meta["evaluation_size"]
    degree_of_dynamicity = payload.get("degree_of_dynamicity", meta["degree_of_dynamicity"])
    cutoff_time = payload.get("cutoff_time", meta["cutoff_time"])
    seed = payload.get("seed") or meta["seed"] or seed_from_text(path.name)
    model_type = payload.get("model_type") or default_model_type
    model_name = payload.get("model_name") or meta["model_name"]
    strategy = parse_strategy(payload.get("strategy") or meta["strategy"], path)
    model_family, model_customer_size = parse_model_family_and_size(model_name)
    solomon_class, solomon_type = parse_solomon_metadata(instance_id)
    scenario_id = payload.get("scenario_id")
    if not scenario_id and default_model_type in {"heuristic", "ai", "hybrid"}:
        scenario_id = build_scenario_id(instance_id, evaluation_size, seed, degree_of_dynamicity, cutoff_time)
    lateness_proxy = payload.get("total_lateness", payload.get("average_lateness"))
    normalized = {
        "run_id": payload.get("run_id"),
        "scenario_id": scenario_id,
        "instance_id": instance_id,
        "evaluation_size": evaluation_size,
        "degree_of_dynamicity": degree_of_dynamicity,
        "cutoff_time": cutoff_time,
        "seed": seed,
        "model_type": model_type,
        "model_name": model_name,
        "strategy": strategy,
        "model_family": model_family,
        "model_customer_size": model_customer_size,
        "allow_rejection": payload.get("allow_rejection"),
        "feasible": payload.get("feasible"),
        "error_message": payload.get("error_message"),
        "num_vehicles": payload.get("num_vehicles"),
        "total_distance": payload.get("total_distance"),
        "total_cost": payload.get("total_cost"),
        "customers_served": payload.get("customers_served"),
        "customers_rejected": payload.get("customers_rejected"),
        "service_level": payload.get("service_level"),
        "rejection_rate": payload.get("rejection_rate"),
        "average_lateness": payload.get("average_lateness"),
        "computational_time": payload.get("computational_time"),
        "routes": payload.get("routes"),
        "timestamp": payload.get("timestamp"),
        "solomon_class": solomon_class,
        "solomon_type": solomon_type,
        "lateness_proxy": lateness_proxy,
        "cost_per_customer": safe_divide(payload.get("total_cost"), max(payload.get("customers_served", 0) or 0, 1), np.nan),
        "distance_per_customer": safe_divide(payload.get("total_distance"), max(payload.get("customers_served", 0) or 0, 1), np.nan),
        "customers_per_vehicle": safe_divide(payload.get("customers_served"), max(payload.get("num_vehicles", 0) or 0, 1), np.nan),
        "method_label": method_label(model_type, strategy, model_family, model_customer_size, model_name),
        "configuration_id": configuration_id(model_type, strategy, model_family, model_customer_size, model_name, payload.get("allow_rejection")),
        "num_reoptimization_events": payload.get("num_reoptimization_events"),
        "runtime_per_customer": safe_divide(payload.get("computational_time"), max(payload.get("customers_served", 0) or 0, 1), np.nan),
        "runtime_per_event": safe_divide(payload.get("computational_time"), max(payload.get("num_reoptimization_events", 0) or 0, 1), np.nan),
        "total_lateness": payload.get("total_lateness"),
        "lateness_count": payload.get("lateness_count"),
        "max_lateness": payload.get("max_lateness"),
        "mean_lateness": payload.get("mean_lateness"),
        "source_file": str(path.resolve()),
    }
    return normalized


def load_dynamic_results_df() -> pd.DataFrame:
    rows = []
    for modality in ["heuristic", "ai", "hybrid"]:
        for path in discover_result_files(modality):
            rows.append(normalize_result_payload(read_json(path), path, modality))
    df = pd.DataFrame(rows)
    if df.empty:
        return empty_df(DYNAMIC_RESULT_COLUMNS)
    df = maybe_numeric(
        df,
        [
            "run_id",
            "evaluation_size",
            "degree_of_dynamicity",
            "cutoff_time",
            "seed",
            "num_vehicles",
            "total_distance",
            "total_cost",
            "customers_served",
            "customers_rejected",
            "service_level",
            "rejection_rate",
            "average_lateness",
            "computational_time",
            "lateness_proxy",
            "cost_per_customer",
            "distance_per_customer",
            "customers_per_vehicle",
            "num_reoptimization_events",
            "runtime_per_customer",
            "runtime_per_event",
            "total_lateness",
            "lateness_count",
            "max_lateness",
            "mean_lateness",
        ],
    )
    df = maybe_datetime(df, ["timestamp"])
    return ensure_columns(df, DYNAMIC_RESULT_COLUMNS)


def load_static_results_df() -> pd.DataFrame:
    rows = []
    if STATIC_RESULTS_ROOT.exists():
        for path in sorted(STATIC_RESULTS_ROOT.glob("*.json")):
            payload = read_json(path)
            base = normalize_result_payload(payload, path, "static")
            row = {
                "static_instance_id": build_static_instance_id(base["instance_id"], base["evaluation_size"]),
                "instance_id": base["instance_id"],
                "evaluation_size": base["evaluation_size"],
                "model_type": base["model_type"],
                "model_name": base["model_name"],
                "strategy": base["strategy"],
                "model_family": base["model_family"],
                "model_customer_size": base["model_customer_size"],
                "feasible": base["feasible"],
                "num_vehicles": base["num_vehicles"],
                "total_distance": base["total_distance"],
                "total_cost": base["total_cost"],
                "customers_served": base["customers_served"],
                "customers_rejected": base["customers_rejected"],
                "service_level": base["service_level"],
                "rejection_rate": base["rejection_rate"],
                "average_lateness": base["average_lateness"],
                "computational_time": base["computational_time"],
                "routes": base["routes"],
                "timestamp": base["timestamp"],
                "solomon_class": base["solomon_class"],
                "solomon_type": base["solomon_type"],
                "method_label": base["method_label"],
                "configuration_id": base["configuration_id"],
                "cost_per_customer": base["cost_per_customer"],
                "distance_per_customer": base["distance_per_customer"],
                "customers_per_vehicle": base["customers_per_vehicle"],
                "source_file": base["source_file"],
            }
            rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return empty_df(STATIC_RESULT_COLUMNS)
    df = maybe_numeric(df, ["evaluation_size", "num_vehicles", "total_distance", "total_cost", "customers_served", "customers_rejected", "service_level", "rejection_rate", "average_lateness", "computational_time", "cost_per_customer", "distance_per_customer", "customers_per_vehicle"])
    df = maybe_datetime(df, ["timestamp"])
    return ensure_columns(df, STATIC_RESULT_COLUMNS)


def load_oracle_results_df() -> pd.DataFrame:
    rows = []
    if ORACLE_RESULTS_ROOT.exists():
        for path in sorted(ORACLE_RESULTS_ROOT.glob("*.json")):
            payload = read_json(path)
            meta = parse_filename_tokens(path)
            instance_id = payload.get("instance_id") or meta["instance_id"]
            evaluation_size = payload.get("evaluation_size") or meta["evaluation_size"]
            solomon_class, solomon_type = parse_solomon_metadata(instance_id)
            rows.append(
                {
                    "instance_id": instance_id,
                    "evaluation_size": evaluation_size,
                    "feasible": payload.get("feasible"),
                    "num_vehicles": payload.get("num_vehicles"),
                    "total_distance": payload.get("total_distance"),
                    "total_cost": payload.get("total_cost"),
                    "computational_time": payload.get("computational_time"),
                    "solomon_class": solomon_class,
                    "solomon_type": solomon_type,
                    "source_file": str(path.resolve()),
                }
            )
    df = pd.DataFrame(rows)
    if df.empty:
        return empty_df(ORACLE_RESULT_COLUMNS)
    df = maybe_numeric(df, ["evaluation_size", "num_vehicles", "total_distance", "total_cost", "computational_time"])
    return ensure_columns(df, ORACLE_RESULT_COLUMNS)


def aggregate_metrics(df: pd.DataFrame, group_cols: list[str], metrics: list[str], sorts: list[str] | None = None) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=group_cols)
    agg = {}
    for metric in metrics:
        if metric in df.columns:
            agg[f"mean_{metric}"] = (metric, "mean")
            agg[f"median_{metric}"] = (metric, "median")
            agg[f"std_{metric}"] = (metric, "std")
    if "scenario_id" in df.columns:
        agg["rows"] = ("scenario_id", "count")
    else:
        agg["rows"] = (group_cols[0], "count")
    grouped = df.groupby(group_cols, dropna=False, as_index=False).agg(**agg)
    if sorts:
        grouped = grouped.sort_values(sorts, kind="stable")
    return grouped.reset_index(drop=True)


def add_dynamic_metrics(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    out = df.copy()
    out["rejection_rate"] = out.apply(lambda row: safe_divide(row["customers_rejected"], row["evaluation_size"], np.nan), axis=1)
    out["service_level"] = out.apply(lambda row: safe_divide(row["customers_served"], row["evaluation_size"], np.nan), axis=1)
    out["cost_per_customer"] = out.apply(lambda row: safe_divide(row["total_cost"], max(row["customers_served"] or 0, 1), np.nan), axis=1)
    out["distance_per_customer"] = out.apply(lambda row: safe_divide(row["total_distance"], max(row["customers_served"] or 0, 1), np.nan), axis=1)
    out["customers_per_vehicle"] = out.apply(lambda row: safe_divide(row["customers_served"], max(row["num_vehicles"] or 0, 1), np.nan), axis=1)
    out["runtime_per_customer"] = out.apply(lambda row: safe_divide(row["computational_time"], max(row["customers_served"] or 0, 1), np.nan), axis=1)
    if "num_reoptimization_events" in out.columns:
        out["runtime_per_event"] = out.apply(lambda row: safe_divide(row["computational_time"], max(row["num_reoptimization_events"] or 0, 1), np.nan), axis=1)
    if "total_lateness" in out.columns and "lateness_count" in out.columns:
        out["mean_lateness"] = out.apply(lambda row: safe_divide(row["total_lateness"], max(row["lateness_count"] or 0, 1), np.nan), axis=1)
    return out


def expected_dynamic_customers(evaluation_size, degree_of_dynamicity):
    if pd.isna(evaluation_size) or pd.isna(degree_of_dynamicity):
        return np.nan
    return int(round(float(evaluation_size) * float(degree_of_dynamicity)))


def coverage_summary_for_modality(df: pd.DataFrame, model_type: str) -> pd.DataFrame:
    subset = df[df["model_type"].eq(model_type)].copy()
    if subset.empty:
        return pd.DataFrame([{"model_type": model_type, "observed_rows": 0, "expected_rows": 0, "coverage_ratio": 0.0}])
    grouped = subset.groupby(["model_type", "configuration_id"], dropna=False, as_index=False).agg(
        observed_rows=("scenario_id", "count"),
        scenarios=("scenario_id", "nunique"),
    )
    grouped["expected_rows"] = EXPECTED_SCENARIO_COUNT
    grouped["coverage_ratio"] = grouped["observed_rows"] / grouped["expected_rows"]
    return grouped.sort_values(["configuration_id"], kind="stable").reset_index(drop=True)


def scenario_heatmap(df: pd.DataFrame, value_col: str, title: str, cmap: str = "Blues"):
    if df.empty:
        return placeholder_figure(title, "No data available.")
    fig, ax = plt.subplots(figsize=(6, 4))
    pivot = df.pivot(index="degree_of_dynamicity", columns="cutoff_time", values=value_col).sort_index().sort_index(axis=1)
    if sns is not None:
        sns.heatmap(pivot, annot=True, fmt=".0f", cmap=cmap, ax=ax)
    else:
        im = ax.imshow(pivot.fillna(0).values, cmap=cmap, aspect="auto")
        fig.colorbar(im, ax=ax)
        ax.set_xticks(range(len(pivot.columns)), [str(value) for value in pivot.columns])
        ax.set_yticks(range(len(pivot.index)), [str(value) for value in pivot.index])
    ax.set_title(title)
    ax.set_xlabel("cutoff_time")
    ax.set_ylabel("degree_of_dynamicity")
    return fig, ax


def boxplot_or_placeholder(df: pd.DataFrame, x: str, y: str, title: str, hue: str | None = None, figsize=(10, 5)):
    if df.empty or x not in df.columns or y not in df.columns or df[y].dropna().empty:
        return placeholder_figure(title, "No non-empty data available for this plot.", figsize=figsize)
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.boxplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        categories = [str(value) for value in sorted(df[x].dropna().unique())]
        data = [df[df[x].astype(str).eq(category)][y].dropna().values for category in categories]
        ax.boxplot(data, labels=categories)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
    return fig, ax


def barplot_or_placeholder(df: pd.DataFrame, x: str, y: str, title: str, hue: str | None = None, figsize=(10, 5)):
    if df.empty or x not in df.columns or y not in df.columns or df[y].dropna().empty:
        return placeholder_figure(title, "No non-empty data available for this plot.", figsize=figsize)
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax, errorbar=None)
    else:
        plot_df = df.copy()
        ax.bar(plot_df[x].astype(str), plot_df[y])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
    return fig, ax


def scatter_or_placeholder(df: pd.DataFrame, x: str, y: str, title: str, hue: str | None = None, figsize=(8, 6)):
    if df.empty or x not in df.columns or y not in df.columns or df[[x, y]].dropna().empty:
        return placeholder_figure(title, "No paired x/y observations available.", figsize=figsize)
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.scatterplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        ax.scatter(df[x], df[y], s=30)
    ax.set_title(title)
    ax.grid(alpha=0.2)
    return fig, ax


def hist_or_placeholder(df: pd.DataFrame, x: str, title: str, hue: str | None = None, figsize=(10, 5)):
    if df.empty or x not in df.columns or df[x].dropna().empty:
        return placeholder_figure(title, "No observations available.", figsize=figsize)
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.histplot(data=df, x=x, hue=hue, kde=False, ax=ax)
    else:
        ax.hist(df[x].dropna(), bins=20)
    ax.set_title(title)
    return fig, ax


def lineplot_or_placeholder(df: pd.DataFrame, x: str, y: str, title: str, hue: str | None = None, style: str | None = None, figsize=(10, 5)):
    if df.empty or x not in df.columns or y not in df.columns or df[y].dropna().empty:
        return placeholder_figure(title, "No observations available.", figsize=figsize)
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.lineplot(data=df, x=x, y=y, hue=hue, style=style, marker="o", ax=ax)
    else:
        for key, part in df.groupby(hue or x):
            ax.plot(part[x], part[y], marker="o", label=str(key))
        if hue:
            ax.legend()
    ax.set_title(title)
    ax.grid(alpha=0.2)
    return fig, ax


def build_pairwise_gap_df(left: pd.DataFrame, right: pd.DataFrame, join_keys: list[str], prefix: str, left_name: str, right_name: str) -> pd.DataFrame:
    if left.empty or right.empty:
        return pd.DataFrame()
    merged = left.merge(right, on=join_keys, how="inner", suffixes=(f"_{left_name}", f"_{right_name}"))
    if merged.empty:
        return merged
    merged[f"{prefix}_%"] = (merged[f"total_cost_{left_name}"] - merged[f"total_cost_{right_name}"]) / merged[f"total_cost_{right_name}"] * 100.0
    merged[f"distance_{prefix}_%"] = (merged[f"total_distance_{left_name}"] - merged[f"total_distance_{right_name}"]) / merged[f"total_distance_{right_name}"] * 100.0
    merged[f"vehicle_difference_{prefix}"] = merged[f"num_vehicles_{left_name}"] - merged[f"num_vehicles_{right_name}"]
    merged[f"service_level_difference_{prefix}"] = merged[f"service_level_{left_name}"] - merged[f"service_level_{right_name}"]
    merged[f"rejection_difference_{prefix}"] = merged[f"rejection_rate_{left_name}"] - merged[f"rejection_rate_{right_name}"]
    merged[f"runtime_ratio_{prefix}"] = merged[f"computational_time_{left_name}"] / merged[f"computational_time_{right_name}"].replace({0: np.nan})
    merged[f"win_{prefix}"] = merged[f"total_cost_{left_name}"] < merged[f"total_cost_{right_name}"]
    return merged


def average_rank_per_scenario(df: pd.DataFrame, score_col: str, group_cols: list[str], ascending: bool = True) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=group_cols + ["average_rank_per_scenario"])
    ranked = df.copy()
    ranked["scenario_rank"] = ranked.groupby("scenario_id")[score_col].rank(method="average", ascending=ascending)
    out = ranked.groupby(group_cols, dropna=False, as_index=False).agg(average_rank_per_scenario=("scenario_rank", "mean"))
    return out


def bootstrap_ci(values, func=np.mean, n_boot=1000, ci=95):
    array = pd.Series(values).dropna().to_numpy(dtype=float)
    if len(array) == 0:
        return np.nan, np.nan
    if len(array) == 1:
        return array[0], array[0]
    samples = []
    rng = np.random.default_rng(42)
    for _ in range(n_boot):
        sample = rng.choice(array, size=len(array), replace=True)
        samples.append(func(sample))
    alpha = (100 - ci) / 2
    return float(np.percentile(samples, alpha)), float(np.percentile(samples, 100 - alpha))


def paired_test_table(df: pd.DataFrame, value_col: str, left_filter: pd.Series, right_filter: pd.Series, label: str) -> pd.DataFrame:
    left = df[left_filter][["scenario_id", "configuration_id", value_col]].rename(columns={value_col: "left_value"})
    right = df[right_filter][["scenario_id", "configuration_id", value_col]].rename(columns={value_col: "right_value"})
    merged = left.merge(right, on="scenario_id", how="inner")
    if merged.empty:
        return pd.DataFrame([{"comparison": label, "n_pairs": 0, "statistic": np.nan, "p_value": np.nan, "effect_size": np.nan, "ci_low": np.nan, "ci_high": np.nan}])
    diff = merged["left_value"] - merged["right_value"]
    if stats is not None and len(diff.dropna()) > 0:
        try:
            result = stats.wilcoxon(diff.dropna())
            statistic = float(result.statistic)
            p_value = float(result.pvalue)
        except Exception:
            statistic = np.nan
            p_value = np.nan
    else:
        statistic = np.nan
        p_value = np.nan
    effect_size = safe_divide(diff.mean(), diff.std(ddof=0), np.nan)
    ci_low, ci_high = bootstrap_ci(diff.dropna())
    return pd.DataFrame(
        [
            {
                "comparison": label,
                "n_pairs": int(diff.dropna().shape[0]),
                "statistic": statistic,
                "p_value": p_value,
                "effect_size": effect_size,
                "ci_low": ci_low,
                "ci_high": ci_high,
            }
        ]
    )


def one_sample_signed_rank_table(values, label: str) -> pd.DataFrame:
    series = pd.Series(values).dropna()
    if series.empty:
        return pd.DataFrame([{"comparison": label, "n_pairs": 0, "statistic": np.nan, "p_value": np.nan, "effect_size": np.nan, "ci_low": np.nan, "ci_high": np.nan}])
    if stats is not None:
        try:
            result = stats.wilcoxon(series)
            statistic = float(result.statistic)
            p_value = float(result.pvalue)
        except Exception:
            statistic = np.nan
            p_value = np.nan
    else:
        statistic = np.nan
        p_value = np.nan
    effect_size = safe_divide(series.mean(), series.std(ddof=0), np.nan)
    ci_low, ci_high = bootstrap_ci(series)
    return pd.DataFrame(
        [
            {
                "comparison": label,
                "n_pairs": int(series.shape[0]),
                "statistic": statistic,
                "p_value": p_value,
                "effect_size": effect_size,
                "ci_low": ci_low,
                "ci_high": ci_high,
            }
        ]
    )


def best_config_table(df: pd.DataFrame, group_cols: list[str], score_col: str) -> pd.DataFrame:
    if df.empty or score_col not in df.columns:
        return pd.DataFrame(columns=group_cols + ["configuration_id", score_col])
    sort_cols = group_cols + [score_col, "mean_rejection_rate", "mean_average_lateness", "mean_computational_time"]
    present = [column for column in sort_cols if column in df.columns]
    ranked = df.sort_values(present, kind="stable").copy()
    if not group_cols:
        return ranked.head(1).reset_index(drop=True)
    return ranked.groupby(group_cols, dropna=False, as_index=False).first()


def placeholder_table(label: str, columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame([{column: pd.NA for column in columns}] if columns else [])


## 1. Load and normalise scenario definitions


In [3]:
scenarios_df = load_scenarios_df()
print_table("scenarios_df", scenarios_df)


scenarios_df: 3024 rows x 21 columns


,scenario_id,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,seed,dynamic_customer_ids,reveal_times,feasible,dropped_reason,solomon_class,solomon_type,num_dynamic_customers,num_static_customers,min_reveal_time,mean_reveal_time,max_reveal_time,reveal_time_std,mean_reveal_fraction,max_reveal_fraction,source_file
0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
1,C101_n50_seed3765530_dod0p3_cut0p5,C101.txt,50,0.3,0.5,3765530,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[547.6407110584447, 24.3843598096617, 270.6860...",True,None,C,1,15,35.0,11.001816,169.944562,547.640711,178.821095,0.150794,0.485928,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
2,C101_n50_seed3765530_dod0p3_cut0p8,C101.txt,50,0.3,0.8,3765530,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[770.9505155677134, 24.3843598096617, 342.5186...",True,None,C,1,15,35.0,11.001816,211.865916,771.349068,251.353580,0.187991,0.684427,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
3,C101_n50_seed3765530_dod0p5_cut0p2,C101.txt,50,0.5,0.2,3765530,"[2, 3, 4, 8, 9, 18, 20, 21, 22, 25, 26, 29, 34...","[7.905955957829787, 29.20278196426196, 179.238...",True,None,C,1,25,25.0,4.205021,108.872318,246.718086,69.334060,0.096604,0.218916,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
4,C101_n50_seed3765530_dod0p5_cut0p5,C101.txt,50,0.5,0.5,3765530,"[2, 3, 4, 8, 9, 18, 20, 21, 22, 25, 26, 29, 34...","[19.764889894574466, 29.20278196426196, 448.09...",True,None,C,1,25,25.0,4.205021,204.790214,505.927854,139.207352,0.181713,0.448916,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
5,C101_n50_seed3765530_dod0p5_cut0p8,C101.txt,50,0.5,0.8,3765530,"[2, 3, 4, 8, 9, 18, 20, 21, 22, 25, 26, 29, 34...","[27.824359560323273, 29.20278196426196, 567.00...",True,None,C,1,25,25.0,4.205021,251.062212,722.871028,198.486459,0.222770,0.641412,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
6,C101_n50_seed3765530_dod0p7_cut0p2,C101.txt,50,0.7,0.2,3765530,"[1, 2, 3, 6, 9, 10, 12, 13, 14, 15, 16, 19, 20...","[95.69069156097063, 97.05852571070132, 69.4672...",True,None,C,1,35,15.0,5.690764,106.233153,246.718086,66.600214,0.094262,0.218916,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
7,C101_n50_seed3765530_dod0p7_cut0p5,C101.txt,50,0.7,0.5,3765530,"[1, 2, 3, 6, 9, 10, 12, 13, 14, 15, 16, 19, 20...","[239.22672890242657, 242.6463142767533, 69.467...",True,None,C,1,35,15.0,14.226909,195.234139,616.795216,149.574152,0.173233,0.547289,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
8,C101_n50_seed3765530_dod0p7_cut0p8,C101.txt,50,0.7,0.8,3765530,"[1, 2, 3, 6, 9, 10, 12, 13, 14, 15, 16, 19, 20...","[374.32402402693606, 341.5894715546527, 69.467...",True,None,C,1,35,15.0,20.258382,219.884426,700.631459,173.960014,0.195106,0.621678,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
9,C101_n50_seed851657_dod0p3_cut0p2,C101.txt,50,0.3,0.2,851657,"[3, 4, 6, 7, 11, 14, 17, 21, 26, 30, 34, 35, 3...","[83.57094805186338, 226.5204578255235, 150.832...",True,None,C,1,15,35.0,5.092663,115.770963,227.288344,73.950041,0.102725,0.201676,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...


## 2. Validate scenario generation


In [4]:
scenario_count_by_size_DoD_cutoff = (
    scenarios_df.groupby(["evaluation_size", "degree_of_dynamicity", "cutoff_time"], dropna=False, as_index=False)
    .agg(scenario_count=("scenario_id", "count"))
    .sort_values(["evaluation_size", "degree_of_dynamicity", "cutoff_time"], kind="stable")
)
dynamic_customer_count_summary = (
    scenarios_df.groupby(["evaluation_size", "degree_of_dynamicity"], dropna=False, as_index=False)
    .agg(
        mean_dynamic_customers=("num_dynamic_customers", "mean"),
        min_dynamic_customers=("num_dynamic_customers", "min"),
        max_dynamic_customers=("num_dynamic_customers", "max"),
    )
)
reveal_time_summary_by_cutoff = (
    scenarios_df.groupby(["cutoff_time"], dropna=False, as_index=False)
    .agg(
        mean_reveal_time=("mean_reveal_time", "mean"),
        median_reveal_time=("mean_reveal_time", "median"),
        max_reveal_time=("max_reveal_time", "max"),
    )
)

scenario_validation_checks = pd.DataFrame(
    [
        {"check": "total scenarios", "expected": EXPECTED_SCENARIO_COUNT, "observed": len(scenarios_df), "passed": len(scenarios_df) == EXPECTED_SCENARIO_COUNT},
        {"check": "unique instances", "expected": EXPECTED_INSTANCE_COUNT, "observed": scenarios_df["instance_id"].nunique(), "passed": scenarios_df["instance_id"].nunique() == EXPECTED_INSTANCE_COUNT},
        {"check": "unique scenario_id", "expected": len(scenarios_df), "observed": scenarios_df["scenario_id"].nunique(), "passed": scenarios_df["scenario_id"].nunique() == len(scenarios_df)},
        {"check": "missing reveal times for dynamic customers", "expected": 0, "observed": int(((scenarios_df["num_dynamic_customers"] > 0) & (scenarios_df["reveal_times"].map(len) == 0)).sum()), "passed": int(((scenarios_df["num_dynamic_customers"] > 0) & (scenarios_df["reveal_times"].map(len) == 0)).sum()) == 0},
    ]
)
scenario_validation_checks["notes"] = ""
scenario_validation_checks.loc[scenario_validation_checks["check"].eq("total scenarios") & ~scenario_validation_checks["passed"], "notes"] = "Scenario directory coverage differs from the intended benchmark."

expected_dynamic = scenarios_df.apply(lambda row: expected_dynamic_customers(row["evaluation_size"], row["degree_of_dynamicity"]), axis=1)
scenario_dynamic_customer_check = pd.DataFrame(
    {
        "scenario_id": scenarios_df["scenario_id"],
        "expected_dynamic_customers": expected_dynamic,
        "observed_dynamic_customers": scenarios_df["num_dynamic_customers"],
    }
)
scenario_dynamic_customer_check["matches_expected"] = scenario_dynamic_customer_check["expected_dynamic_customers"] == scenario_dynamic_customer_check["observed_dynamic_customers"]

print_table("scenario_validation_checks", scenario_validation_checks)
print_table("scenario_count_by_size_DoD_cutoff", scenario_count_by_size_DoD_cutoff)
print_table("dynamic_customer_count_summary", dynamic_customer_count_summary)
print_table("reveal_time_summary_by_cutoff", reveal_time_summary_by_cutoff)

heatmap_input = scenarios_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(scenario_count=("scenario_id", "count"))
fig, _ = scenario_heatmap(heatmap_input, "scenario_count", "Scenario count by DoD x cutoff")
save_figure(fig, "scenario_validation", "scenario_count_heatmap.png")

fig, _ = barplot_or_placeholder(dynamic_customer_count_summary, "degree_of_dynamicity", "mean_dynamic_customers", "Dynamic customer count by DoD and size", hue="evaluation_size")
save_figure(fig, "scenario_validation", "dynamic_customer_count_bar.png")

fig, _ = hist_or_placeholder(scenarios_df, "mean_reveal_time", "Reveal time distribution by cutoff", hue="cutoff_time")
save_figure(fig, "scenario_validation", "reveal_time_histogram.png")


scenario_validation_checks: 4 rows x 5 columns


,check,expected,observed,passed,notes
0,total scenarios,3024,3024,True,
1,unique instances,56,56,True,
2,unique scenario_id,3024,3024,True,
3,missing reveal times for dynamic customers,0,0,True,


scenario_count_by_size_DoD_cutoff: 18 rows x 4 columns


,evaluation_size,degree_of_dynamicity,cutoff_time,scenario_count
0,50,0.3,0.2,168
1,50,0.3,0.5,168
2,50,0.3,0.8,168
3,50,0.5,0.2,168
4,50,0.5,0.5,168
5,50,0.5,0.8,168
6,50,0.7,0.2,168
7,50,0.7,0.5,168
8,50,0.7,0.8,168
9,75,0.3,0.2,168


dynamic_customer_count_summary: 6 rows x 5 columns


,evaluation_size,degree_of_dynamicity,mean_dynamic_customers,min_dynamic_customers,max_dynamic_customers
0,50,0.3,15.0,15,15
1,50,0.5,25.0,25,25
2,50,0.7,35.0,35,35
3,75,0.3,22.0,22,22
4,75,0.5,38.0,38,38
5,75,0.7,52.0,52,52


reveal_time_summary_by_cutoff: 3 rows x 4 columns


,cutoff_time,mean_reveal_time,median_reveal_time,max_reveal_time
0,0.2,106.214149,95.707907,676.678246
1,0.5,231.220423,212.403356,1691.695616
2,0.8,308.881941,271.658587,2706.712985


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/scenario_validation/reveal_time_histogram.png')

## 3. Load and normalise dynamic result units


In [5]:
dynamic_results_df = load_dynamic_results_df()
print_table("dynamic_results_df", dynamic_results_df)


dynamic_results_df: 18211 rows x 42 columns


,run_id,scenario_id,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,seed,model_type,model_name,strategy,model_family,model_customer_size,allow_rejection,feasible,error_message,num_vehicles,total_distance,total_cost,customers_served,customers_rejected,service_level,rejection_rate,average_lateness,computational_time,routes,timestamp,solomon_class,solomon_type,lateness_proxy,cost_per_customer,distance_per_customer,customers_per_vehicle,method_label,configuration_id,num_reoptimization_events,runtime_per_customer,runtime_per_event,total_lateness,lateness_count,max_lateness,mean_lateness,source_file
0,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,heuristic,ortools,NA,NA,NA,False,True,None,6,427.0,1027.0,50,0,1.00,0.0,0.00,0.0,"[{'node_ids': [32, 33], 'vehicle_id': 0}, {'no...",2026-04-29 22:29:04.735381,C,1,0.00,20.54,8.54,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
1,0,C101_n50_seed3765530_dod0p3_cut0p5,C101.txt,50,0.3,0.5,3765530,heuristic,ortools,NA,NA,NA,False,True,None,6,426.0,1026.0,50,0,1.00,0.0,0.00,0.0,"[{'node_ids': [32, 33, 30, 28, 26, 23, 21], 'v...",2026-04-29 22:29:04.745386,C,1,0.00,20.52,8.52,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
2,0,C101_n50_seed3765530_dod0p3_cut0p8,C101.txt,50,0.3,0.8,3765530,heuristic,ortools,NA,NA,NA,False,True,None,6,423.0,1023.0,50,0,1.00,0.0,0.00,0.0,"[{'node_ids': [32, 33, 29], 'vehicle_id': 0}, ...",2026-04-29 22:29:04.740385,C,1,0.00,20.46,8.46,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
3,0,C101_n50_seed3765530_dod0p5_cut0p2,C101.txt,50,0.5,0.2,3765530,heuristic,ortools,NA,NA,NA,False,False,None,6,397.0,997.0,50,0,0.98,0.0,0.30,0.0,"[{'node_ids': [42, 41, 40, 44, 46, 45, 48, 50,...",2026-04-29 22:29:54.816519,C,1,0.30,19.94,7.94,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
4,0,C101_n50_seed3765530_dod0p5_cut0p5,C101.txt,50,0.5,0.5,3765530,heuristic,ortools,NA,NA,NA,False,False,None,6,452.0,1052.0,50,0,0.96,0.0,0.32,0.0,"[{'node_ids': [42, 41, 40, 44, 46, 45, 48, 50,...",2026-04-29 22:31:39.916287,C,1,0.32,21.04,9.04,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
5,0,C101_n50_seed3765530_dod0p5_cut0p8,C101.txt,50,0.5,0.8,3765530,heuristic,ortools,NA,NA,NA,False,False,None,6,463.0,1063.0,50,0,0.96,0.0,0.32,0.0,"[{'node_ids': [42, 41, 40, 44, 46, 45, 48], 'v...",2026-04-29 22:31:39.996818,C,1,0.32,21.26,9.26,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
6,0,C101_n50_seed3765530_dod0p7_cut0p2,C101.txt,50,0.7,0.2,3765530,heuristic,ortools,NA,NA,NA,False,True,None,8,541.0,1341.0,50,0,1.00,0.0,0.00,0.0,"[{'node_ids': [13], 'vehicle_id': 0}, {'node_i...",2026-04-29 22:32:30.135772,C,1,0.00,26.82,10.82,6.250000,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
7,0,C101_n50_seed3765530_dod0p7_cut0p5,C101.txt,50,0.7,0.5,3765530,heuristic,ortools,NA,NA,NA,False,True,None,8,546.0,1346.0,50,0,1.00,0.0,0.00,0.0,"[{'node_ids': [13, 15, 16, 14, 12], 'vehicle_i...",2026-04-29 22:33:20.023354,C,1,0.00,26.92,10.92,6.250000,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
8,0,C101_n50_seed3765530_dod0p7_cut0p8,C101.txt,50,0.7,0.8,3765530,heuristic,ortools,NA,NA,NA,False,True,None,9,589.0,1489.0,50,0,1.00,0.0,0.00,0.0,"[{'node_ids': [13, 15, 16, 14, 12], 'vehicle_i...",2026-04-29 22:35:05.115019,C,1,0.00,29.78,11.78,5.555556,HEURI

## 4. Validate dynamic result coverage


In [6]:
dynamic_result_count_by_model_type = (
    dynamic_results_df.groupby(["model_type"], dropna=False, as_index=False)
    .agg(rows=("scenario_id", "count"), scenarios=("scenario_id", "nunique"), configurations=("configuration_id", "nunique"))
)

scenario_join_key = ["instance_id", "evaluation_size", "degree_of_dynamicity", "cutoff_time", "seed"]
scenario_lookup = scenarios_df[["scenario_id", *scenario_join_key]].drop_duplicates()
dynamic_join_probe = dynamic_results_df.merge(
    scenario_lookup,
    on=scenario_join_key,
    how="left",
    suffixes=("", "_scenario"),
)
dynamic_join_probe["scenario_id_resolved"] = dynamic_join_probe["scenario_id"].fillna(dynamic_join_probe["scenario_id_scenario"])
missing_results_by_scenario = scenarios_df[["scenario_id"]].merge(
    dynamic_results_df.groupby("scenario_id", as_index=False).agg(result_rows=("scenario_id", "count")),
    on="scenario_id",
    how="left",
)
missing_results_by_scenario["result_rows"] = missing_results_by_scenario["result_rows"].fillna(0)

duplicate_results = (
    dynamic_results_df.groupby(["scenario_id", "configuration_id"], dropna=False, as_index=False)
    .agg(rows=("scenario_id", "count"))
)
duplicate_results = duplicate_results[duplicate_results["rows"] > 1].reset_index(drop=True)

infeasible_or_error_summary = (
    dynamic_results_df.groupby(["model_type", "configuration_id"], dropna=False, as_index=False)
    .agg(
        rows=("scenario_id", "count"),
        infeasible_rows=("feasible", lambda s: int((~s.fillna(False)).sum())),
        error_rows=("error_message", lambda s: int(s.notna().sum())),
    )
)

dynamic_coverage_checks = pd.DataFrame(
    [
        {"check": "heuristic rows", "expected": EXPECTED_SCENARIO_COUNT, "observed": int(dynamic_results_df["model_type"].eq("heuristic").sum())},
        {"check": "AI rows", "expected": 3 * 8 * EXPECTED_SCENARIO_COUNT, "observed": int(dynamic_results_df["model_type"].eq("ai").sum())},
        {"check": "Hybrid rows", "expected": 3 * 8 * EXPECTED_SCENARIO_COUNT, "observed": int(dynamic_results_df["model_type"].eq("hybrid").sum())},
        {"check": "valid scenario join coverage", "expected": len(dynamic_results_df), "observed": int(dynamic_join_probe["scenario_id_resolved"].notna().sum())},
    ]
)
dynamic_coverage_checks["passed"] = dynamic_coverage_checks["expected"] == dynamic_coverage_checks["observed"]

print_table("dynamic_coverage_checks", dynamic_coverage_checks)
print_table("dynamic_result_count_by_model_type", dynamic_result_count_by_model_type)
print_table("missing_results_by_scenario", missing_results_by_scenario[missing_results_by_scenario["result_rows"].eq(0)])
print_table("duplicate_results", duplicate_results)
print_table("infeasible_or_error_summary", infeasible_or_error_summary)


dynamic_coverage_checks: 4 rows x 4 columns


,check,expected,observed,passed
0,heuristic rows,3024,3024,True
1,AI rows,72576,15183,False
2,Hybrid rows,72576,4,False
3,valid scenario join coverage,18211,18211,True


dynamic_result_count_by_model_type: 3 rows x 4 columns


,model_type,rows,scenarios,configurations
0,ai,15183,3024,10
1,heuristic,3024,3024,1
2,hybrid,4,1,4


missing_results_by_scenario: 0 rows x 2 columns


,scenario_id,result_rows


duplicate_results: 24 rows x 3 columns


,scenario_id,configuration_id,rows
0,C101_n50_seed3765530_dod0p3_cut0p2,ai|greedy|general|50|general_50|reject-0,2
1,C101_n50_seed3765530_dod0p3_cut0p2,ai|greedy|lateness|50|routefinder_with_latenes...,2
2,C101_n50_seed3765530_dod0p3_cut0p2,ai|greedy|lateness|75|routefinder_with_latenes...,2
3,C101_n50_seed3765530_dod0p3_cut0p2,ai|greedy|solomon|50|routefinder_solomon_gener...,2
4,C101_n50_seed3765530_dod0p3_cut0p2,ai|greedy|solomon|75|routefinder_solomon_gener...,2
5,C101_n50_seed3765530_dod0p3_cut0p5,ai|greedy|general|50|general_50|reject-0,2
6,C101_n50_seed3765530_dod0p3_cut0p5,ai|greedy|lateness|50|routefinder_with_latenes...,2
7,C101_n50_seed3765530_dod0p3_cut0p5,ai|greedy|lateness|75|routefinder_with_latenes...,2
8,C101_n50_seed3765530_dod0p3_cut0p5,ai|greedy|solomon|50|routefinder_solomon_gener...,2
9,C101_n50_seed3765530_dod0p3_cut0p5,ai|greedy|solomon|75|routefinder_solomon_gener...,2


infeasible_or_error_summary: 15 rows x 5 columns


,model_type,configuration_id,rows,infeasible_rows,error_rows
0,ai,ai|NA|general|50|general_50|reject-0,3017,2831,0
1,ai,ai|NA|lateness|50|routefinder_with_lateness_50...,3018,2814,0
2,ai,ai|NA|lateness|75|routefinder_with_lateness_75...,3018,2835,0
3,ai,ai|NA|solomon|50|routefinder_solomon_generated...,3017,2813,0
4,ai,ai|NA|solomon|75|routefinder_solomon_generated...,3017,2737,0
5,ai,ai|greedy|general|50|general_50|reject-0,20,20,0
6,ai,ai|greedy|lateness|50|routefinder_with_latenes...,19,18,0
7,ai,ai|greedy|lateness|75|routefinder_with_latenes...,18,18,0
8,ai,ai|greedy|solomon|50|routefinder_solomon_gener...,20,20,0
9,ai,ai|greedy|solomon|75|routefinder_solomon_gener...,19,19,0


## 5. Join dynamic results with scenario metadata

## 6. Compute base dynamic metrics


In [7]:
scenario_join_key = ["instance_id", "evaluation_size", "degree_of_dynamicity", "cutoff_time", "seed"]
dynamic_eval_df = dynamic_results_df.merge(
    scenarios_df.drop(columns=["instance_id", "evaluation_size", "degree_of_dynamicity", "cutoff_time", "seed"], errors="ignore"),
    on="scenario_id",
    how="left",
    suffixes=("", "_scenario"),
)

missing_join_mask = dynamic_eval_df["source_file_scenario"].isna() if "source_file_scenario" in dynamic_eval_df.columns else dynamic_eval_df["solomon_class"].isna()
if missing_join_mask.any():
    fallback_join = dynamic_eval_df.loc[missing_join_mask].drop(columns=[column for column in dynamic_eval_df.columns if column.endswith("_scenario")], errors="ignore").merge(
        scenarios_df,
        on=scenario_join_key,
        how="left",
        suffixes=("", "_scenario"),
    )
    dynamic_eval_df = pd.concat([dynamic_eval_df.loc[~missing_join_mask], fallback_join], ignore_index=True, sort=False)

if "scenario_id_scenario" in dynamic_eval_df.columns:
    dynamic_eval_df["scenario_id"] = dynamic_eval_df["scenario_id"].fillna(dynamic_eval_df["scenario_id_scenario"])
dynamic_eval_df = add_dynamic_metrics(dynamic_eval_df)
print_table("dynamic_eval_df", dynamic_eval_df)


dynamic_eval_df: 18211 rows x 57 columns


,run_id,scenario_id,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,seed,model_type,model_name,strategy,model_family,model_customer_size,allow_rejection,feasible,error_message,num_vehicles,total_distance,total_cost,customers_served,customers_rejected,service_level,rejection_rate,average_lateness,computational_time,routes,timestamp,solomon_class,solomon_type,lateness_proxy,cost_per_customer,distance_per_customer,customers_per_vehicle,method_label,configuration_id,num_reoptimization_events,runtime_per_customer,runtime_per_event,total_lateness,lateness_count,max_lateness,mean_lateness,source_file,dynamic_customer_ids,reveal_times,feasible_scenario,dropped_reason,solomon_class_scenario,solomon_type_scenario,num_dynamic_customers,num_static_customers,min_reveal_time,mean_reveal_time,max_reveal_time,reveal_time_std,mean_reveal_fraction,max_reveal_fraction,source_file_scenario
0,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,heuristic,ortools,NA,NA,NA,False,True,None,6,427.0,1027.0,50,0,1.0,0.0,0.00,0.0,"[{'node_ids': [32, 33], 'vehicle_id': 0}, {'no...",2026-04-29 22:29:04.735381,C,1,0.00,20.54,8.54,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
1,0,C101_n50_seed3765530_dod0p3_cut0p5,C101.txt,50,0.3,0.5,3765530,heuristic,ortools,NA,NA,NA,False,True,None,6,426.0,1026.0,50,0,1.0,0.0,0.00,0.0,"[{'node_ids': [32, 33, 30, 28, 26, 23, 21], 'v...",2026-04-29 22:29:04.745386,C,1,0.00,20.52,8.52,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[547.6407110584447, 24.3843598096617, 270.6860...",True,None,C,1,15,35.0,11.001816,169.944562,547.640711,178.821095,0.150794,0.485928,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
2,0,C101_n50_seed3765530_dod0p3_cut0p8,C101.txt,50,0.3,0.8,3765530,heuristic,ortools,NA,NA,NA,False,True,None,6,423.0,1023.0,50,0,1.0,0.0,0.00,0.0,"[{'node_ids': [32, 33, 29], 'vehicle_id': 0}, ...",2026-04-29 22:29:04.740385,C,1,0.00,20.46,8.46,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[770.9505155677134, 24.3843598096617, 342.5186...",True,None,C,1,15,35.0,11.001816,211.865916,771.349068,251.353580,0.187991,0.684427,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
3,0,C101_n50_seed3765530_dod0p5_cut0p2,C101.txt,50,0.5,0.2,3765530,heuristic,ortools,NA,NA,NA,False,False,None,6,397.0,997.0,50,0,1.0,0.0,0.30,0.0,"[{'node_ids': [42, 41, 40, 44, 46, 45, 48, 50,...",2026-04-29 22:29:54.816519,C,1,0.30,19.94,7.94,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 8, 9, 18, 20, 21, 22, 25, 26, 29, 34...","[7.905955957829787, 29.20278196426196, 179.238...",True,None,C,1,25,25.0,4.205021,108.872318,246.718086,69.334060,0.096604,0.218916,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
4,0,C101_n50_seed3765530_dod0p5_cut0p5,C101.txt,50,0.5,0.5,3765530,heuristic,ortools,NA,NA,NA,False,False,None,6,452.0,1052.0,50,0,1.0,0.0,0.32,0.0,"[{'node_ids': [42, 41, 40, 44, 46, 45, 48, 50,...",2026-04-29 22:31:39.916287,C,1,0.32,21.04,9.04,8.333333,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 8, 9, 18, 20, 21, 22, 25, 26, 29, 34...","[19.764889894574466, 29.20278196426196, 448.09...",True,None,C,1,25,25.0,4.205021,204.790214,505.927854,139.207352,0.181713,0.448916,/Users/giuseppe/Documents/per

## 7. Load and normalise static benchmark results

## 8. Load and normalise static oracle results


In [8]:
static_results_df = load_static_results_df()
oracle_results_df = load_oracle_results_df()
print_table("static_results_df", static_results_df)
print_table("oracle_results_df", oracle_results_df)


static_results_df: 2352 rows x 28 columns


,static_instance_id,instance_id,evaluation_size,model_type,model_name,strategy,model_family,model_customer_size,feasible,num_vehicles,total_distance,total_cost,customers_served,customers_rejected,service_level,rejection_rate,average_lateness,computational_time,routes,timestamp,solomon_class,solomon_type,method_label,configuration_id,cost_per_customer,distance_per_customer,customers_per_vehicle,source_file
0,C101_n50,C101.txt,50,ai,general_50,NA,general,50,True,6,373.0,973.0,50,0,1.0,0.0,0.0,0.070412,"[{'node_ids': [32, 33, 31, 35, 37, 38, 39, 36,...",2026-05-01 17:10:11.908001,C,1,AI | general-50,ai|NA|general|50|general_50|reject-0,19.46,7.46,8.333333,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
1,C101_n50,C101.txt,50,ai,general_50,greedy,general,50,True,6,373.0,973.0,50,0,1.0,0.0,0.0,0.071885,"[{'node_ids': [32, 33, 31, 35, 37, 38, 39, 36,...",2026-05-01 17:42:04.318178,C,1,AI | greedy | general-50,ai|greedy|general|50|general_50|reject-0,19.46,7.46,8.333333,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
2,C101_n50,C101.txt,50,ai,general_50,sampling,general,50,True,7,408.0,1108.0,50,0,1.0,0.0,0.0,0.154956,"[{'node_ids': [13, 17, 18, 19, 15, 16, 14, 12]...",2026-05-01 17:38:35.528693,C,1,AI | sampling | general-50,ai|sampling|general|50|general_50|reject-0,22.16,8.16,7.142857,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
3,C101_n50,C101.txt,50,ai,general_50,sampling,general,50,True,7,410.0,1110.0,50,0,1.0,0.0,0.0,0.026138,"[{'node_ids': [13, 17, 18, 19, 15, 16, 14, 12]...",2026-05-01 17:36:50.199226,C,1,AI | sampling | general-50,ai|sampling|general|50|general_50|reject-0,22.20,8.20,7.142857,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
4,C101_n50,C101.txt,50,heuristic,ortools,NA,NA,NA,True,5,353.0,853.0,50,0,1.0,0.0,0.0,5.015944,"[{'node_ids': [], 'vehicle_id': 0}, {'node_ids...",2026-05-01 17:10:09.899305,C,1,HEURISTIC | ortools,heuristic|NA|NA|NA|ortools|reject-0,17.06,7.06,10.000000,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
5,C101_n50,C101.txt,50,ai,routefinder_solomon_generated_50,NA,solomon,50,True,14,682.0,2082.0,50,0,1.0,0.0,0.0,0.086431,"[{'node_ids': [13, 17, 18, 19, 15, 16, 14, 12]...",2026-05-01 17:10:13.946054,C,1,AI | solomon-50,ai|NA|solomon|50|routefinder_solomon_generated...,41.64,13.64,3.571429,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
6,C101_n50,C101.txt,50,ai,routefinder_solomon_generated_50,greedy,solomon,50,True,14,682.0,2082.0,50,0,1.0,0.0,0.0,0.087474,"[{'node_ids': [13, 17, 18, 19, 15, 16, 14, 12]...",2026-05-01 17:42:06.540459,C,1,AI | greedy | solomon-50,ai|greedy|solomon|50|routefinder_solomon_gener...,41.64,13.64,3.571429,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
7,C101_n50,C101.txt,50,ai,routefinder_solomon_generated_50,sampling,solomon,50,True,14,780.0,2180.0,50,0,1.0,0.0,0.0,0.201866,"[{'node_ids': [42, 41, 44, 45, 48, 50, 49, 47]...",2026-05-01 17:38:37.310384,C,1,AI | sampling | solomon-50,ai|sampling|solomon|50|routefinder_solomon_gen...,43.60,15.60,3.571429,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
8,C101_n50,C101.txt,50,ai,routefinder_solomon_generated_50,sampling,solomon,50,True,19,1083.0,2983.0,50,0,1.0,0.0,0.0,0.022628,"[{'node_ids': [42, 41, 44, 46, 45, 48, 50, 49,...",2026-05-01 17:36:51.776323,C,1,AI | sampling | solomon-50,ai|sampling|solomon|50|routefinder_solomon_gen...,59.66,21.66,2.631579,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
9,C101_n50,C101.txt,50,ai,routefinder_solomon_generated_75,NA,solomon,75,True,17,944.0,2644.0,50,0,1.0,0.0,0.0,0.098727,"[{'node_ids': [31, 35, 37, 38, 39, 36, 34], 'v...",2026-05-01 17:10:17.215231,C,1,AI | solomon-75,ai|NA|solomon|75|routefinder_solomon_generated...,52.88,18.88,2.941176,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...


oracle_results_df: 56 rows x 10 columns


,instance_id,evaluation_size,feasible,num_vehicles,total_distance,total_cost,computational_time,solomon_class,solomon_type,source_file
0,C101.txt,50,True,5,353.0,853.0,10.006672,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
1,C102.txt,50,True,5,352.0,852.0,10.006134,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
2,C103.txt,50,True,5,353.0,853.0,10.001355,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
3,C104.txt,50,True,5,347.0,847.0,10.001096,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
4,C105.txt,50,True,5,353.0,853.0,10.001128,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
5,C106.txt,50,True,5,353.0,853.0,10.001244,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
6,C107.txt,50,True,5,353.0,853.0,10.001153,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
7,C108.txt,50,True,5,352.0,852.0,10.001090,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
8,C109.txt,50,True,5,352.0,852.0,10.001264,C,1,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...
9,C201.txt,50,True,3,347.0,647.0,10.001210,C,2,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...


## 9. Static benchmark analysis


In [9]:
static_vs_oracle_df = static_results_df.merge(
    oracle_results_df,
    on=["instance_id", "evaluation_size"],
    how="left",
    suffixes=("_static", "_oracle"),
)
static_vs_oracle_df["gap_to_static_oracle_%"] = (static_vs_oracle_df["total_cost_static"] - static_vs_oracle_df["total_cost_oracle"]) / static_vs_oracle_df["total_cost_oracle"] * 100.0
static_vs_oracle_df["distance_gap_to_static_oracle_%"] = (static_vs_oracle_df["total_distance_static"] - static_vs_oracle_df["total_distance_oracle"]) / static_vs_oracle_df["total_distance_oracle"] * 100.0
static_vs_oracle_df["vehicle_difference_to_static_oracle"] = static_vs_oracle_df["num_vehicles_static"] - static_vs_oracle_df["num_vehicles_oracle"]
static_vs_oracle_df["runtime_ratio_to_oracle"] = static_vs_oracle_df["computational_time_static"] / static_vs_oracle_df["computational_time_oracle"].replace({0: np.nan})
static_solomon_col = "solomon_class_static" if "solomon_class_static" in static_vs_oracle_df.columns else "solomon_class"

static_summary_by_method = aggregate_metrics(static_results_df, ["model_type", "strategy", "model_family", "model_customer_size", "model_name"], ["total_cost", "total_distance", "num_vehicles", "service_level", "rejection_rate", "average_lateness", "computational_time"])
static_gap_to_oracle_by_method = aggregate_metrics(static_vs_oracle_df, ["model_type", "strategy", "model_family", "model_customer_size", "model_name"], ["gap_to_static_oracle_%", "distance_gap_to_static_oracle_%", "vehicle_difference_to_static_oracle", "runtime_ratio_to_oracle"])
static_gap_to_oracle_by_solomon_class = aggregate_metrics(static_vs_oracle_df, [static_solomon_col], ["gap_to_static_oracle_%", "distance_gap_to_static_oracle_%"])
static_gap_to_oracle_by_customer_size = aggregate_metrics(static_vs_oracle_df, ["evaluation_size"], ["gap_to_static_oracle_%", "distance_gap_to_static_oracle_%"])

print_table("static_summary_by_method", static_summary_by_method)
print_table("static_gap_to_oracle_by_method", static_gap_to_oracle_by_method)
print_table("static_gap_to_oracle_by_solomon_class", static_gap_to_oracle_by_solomon_class)
print_table("static_gap_to_oracle_by_customer_size", static_gap_to_oracle_by_customer_size)

fig, _ = boxplot_or_placeholder(static_vs_oracle_df, "model_name", "gap_to_static_oracle_%", "Static gap to oracle by method/configuration")
save_figure(fig, "static_oracle", "static_gap_boxplot.png")

fig, _ = scatter_or_placeholder(static_vs_oracle_df, "total_cost_oracle", "total_cost_static", "Oracle total cost vs static total cost", hue="model_name")
save_figure(fig, "static_oracle", "oracle_vs_static_scatter.png")

mean_gap_bar = static_vs_oracle_df.groupby("model_name", as_index=False).agg(mean_gap=("gap_to_static_oracle_%", "mean"))
fig, _ = barplot_or_placeholder(mean_gap_bar, "model_name", "mean_gap", "Mean static gap to oracle by method")
save_figure(fig, "static_oracle", "mean_static_gap_bar.png")


static_summary_by_method: 16 rows x 27 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_total_cost,median_total_cost,std_total_cost,mean_total_distance,median_total_distance,std_total_distance,mean_num_vehicles,median_num_vehicles,std_num_vehicles,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_average_lateness,median_average_lateness,std_average_lateness,mean_computational_time,median_computational_time,std_computational_time,rows
0,ai,NA,general,50,general_50,2192.651786,2096.5,767.112284,1151.580357,1092.5,399.802511,10.410714,10.0,4.363141,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.155346,0.160817,0.085861,112
1,ai,NA,lateness,50,routefinder_with_lateness_50,1998.642857,1926.5,703.574924,1057.571429,1011.0,354.321423,9.410714,9.0,4.028175,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.144880,0.168981,0.067136,112
2,ai,NA,lateness,75,routefinder_with_lateness_75,2136.678571,1945.0,964.682296,1101.857143,1140.5,423.182002,10.348214,9.0,5.833593,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.148358,0.167335,0.070916,112
3,ai,NA,solomon,50,routefinder_solomon_generated_50,2396.955357,2176.0,1052.999605,1151.419643,1093.5,432.105877,12.455357,11.0,6.557972,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.150962,0.178701,0.070896,112
4,ai,NA,solomon,75,routefinder_solomon_generated_75,2284.517857,2198.5,672.484220,1125.589286,1102.5,296.036811,11.589286,11.0,4.228930,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.145210,0.181064,0.064405,112
5,ai,greedy,general,50,general_50,2192.651786,2096.5,767.112284,1151.580357,1092.5,399.802511,10.410714,10.0,4.363141,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.147334,0.164772,0.069007,112
6,ai,greedy,lateness,50,routefinder_with_lateness_50,1998.642857,1926.5,703.574924,1057.571429,1011.0,354.321423,9.410714,9.0,4.028175,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.141227,0.135436,0.065203,112
7,ai,greedy,lateness,75,routefinder_with_lateness_75,2136.678571,1945.0,964.682296,1101.857143,1140.5,423.182002,10.348214,9.0,5.833593,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.142789,0.147380,0.065885,112
8,ai,greedy,solomon,50,routefinder_solomon_generated_50,2396.955357,2176.0,1052.999605,1151.419643,1093.5,432.105877,12.455357,11.0,6.557972,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.147076,0.160075,0.065602,112
9,ai,greedy,solomon,75,routefinder_solomon_generated_75,2284.517857,2198.5,672.484220,1125.589286,1102.5,296.036811,11.589286,11.0,4.228930,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.142304,0.156374,0.060185,112


static_gap_to_oracle_by_method: 16 rows x 18 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_to_static_oracle_%,median_gap_to_static_oracle_%,std_gap_to_static_oracle_%,mean_distance_gap_to_static_oracle_%,median_distance_gap_to_static_oracle_%,std_distance_gap_to_static_oracle_%,mean_vehicle_difference_to_static_oracle,median_vehicle_difference_to_static_oracle,std_vehicle_difference_to_static_oracle,mean_runtime_ratio_to_oracle,median_runtime_ratio_to_oracle,std_runtime_ratio_to_oracle,rows
0,ai,NA,general,50,general_50,77.599019,49.241240,77.041409,70.671934,59.455537,59.269444,3.285714,2.0,3.312315,0.008270,0.007886,0.001511,112
1,ai,NA,lateness,50,routefinder_with_lateness_50,52.732437,43.811031,39.245774,51.653927,44.384922,36.127067,2.357143,2.0,1.862828,0.008318,0.007729,0.002693,112
2,ai,NA,lateness,75,routefinder_with_lateness_75,61.916379,47.987357,53.488988,58.568558,50.039115,43.219469,3.250000,2.0,3.364251,0.008711,0.008024,0.003547,112
3,ai,NA,solomon,50,routefinder_solomon_generated_50,77.198038,75.137295,43.874929,63.752013,56.621215,40.192020,4.607143,5.0,2.800510,0.008539,0.008091,0.002027,112
4,ai,NA,solomon,75,routefinder_solomon_generated_75,89.137804,73.044971,56.123945,75.285813,58.724534,48.007643,4.767857,4.0,2.669634,0.008507,0.008147,0.002526,112
5,ai,greedy,general,50,general_50,77.599019,49.241240,77.041409,70.671934,59.455537,59.269444,3.285714,2.0,3.312315,0.008263,0.007729,0.001767,112
6,ai,greedy,lateness,50,routefinder_with_lateness_50,52.732437,43.811031,39.245774,51.653927,44.384922,36.127067,2.357143,2.0,1.862828,0.007890,0.007788,0.000697,112
7,ai,greedy,lateness,75,routefinder_with_lateness_75,61.916379,47.987357,53.488988,58.568558,50.039115,43.219469,3.250000,2.0,3.364251,0.008150,0.008074,0.001291,112
8,ai,greedy,solomon,50,routefinder_solomon_generated_50,77.198038,75.137295,43.874929,63.752013,56.621215,40.192020,4.607143,5.0,2.800510,0.008448,0.008168,0.001257,112
9,ai,greedy,solomon,75,routefinder_solomon_generated_75,89.137804,73.044971,56.123945,75.285813,58.724534,48.007643,4.767857,4.0,2.669634,0.008398,0.008079,0.001152,112


static_gap_to_oracle_by_solomon_class: 3 rows x 8 columns


,solomon_class_static,mean_gap_to_static_oracle_%,median_gap_to_static_oracle_%,std_gap_to_static_oracle_%,mean_distance_gap_to_static_oracle_%,median_distance_gap_to_static_oracle_%,std_distance_gap_to_static_oracle_%,rows
0,C,121.373634,103.400309,101.435392,118.487307,109.659091,91.802975,714
1,R,78.011782,68.403639,56.156178,61.065450,55.961844,39.551115,966
2,RC,66.862470,47.516433,66.418236,69.113073,51.016949,68.135817,672


static_gap_to_oracle_by_customer_size: 2 rows x 8 columns


,evaluation_size,mean_gap_to_static_oracle_%,median_gap_to_static_oracle_%,std_gap_to_static_oracle_%,mean_distance_gap_to_static_oracle_%,median_distance_gap_to_static_oracle_%,std_distance_gap_to_static_oracle_%,rows
0,50,87.989684,68.557952,78.586864,80.796406,60.678342,71.759473,1176
1,75,NaN,NaN,NaN,NaN,NaN,NaN,1176


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/static_oracle/mean_static_gap_bar.png')

## 10. Dynamic vs heuristic comparison


In [10]:
heuristic_df = dynamic_eval_df[dynamic_eval_df["model_type"].eq("heuristic")].copy()
methods_df = dynamic_eval_df[dynamic_eval_df["model_type"].isin(["ai", "hybrid"])].copy()

dynamic_vs_heuristic_df = methods_df.merge(
    heuristic_df[
        [
            "scenario_id",
            "total_cost",
            "total_distance",
            "num_vehicles",
            "service_level",
            "rejection_rate",
            "computational_time",
            "configuration_id",
        ]
    ].rename(
        columns={
            "total_cost": "heuristic_total_cost",
            "total_distance": "heuristic_total_distance",
            "num_vehicles": "heuristic_num_vehicles",
            "service_level": "heuristic_service_level",
            "rejection_rate": "heuristic_rejection_rate",
            "computational_time": "heuristic_computational_time",
            "configuration_id": "heuristic_configuration_id",
        }
    ),
    on="scenario_id",
    how="left",
)
dynamic_vs_heuristic_df["gap_vs_heuristic_%"] = (dynamic_vs_heuristic_df["total_cost"] - dynamic_vs_heuristic_df["heuristic_total_cost"]) / dynamic_vs_heuristic_df["heuristic_total_cost"] * 100.0
dynamic_vs_heuristic_df["distance_gap_vs_heuristic_%"] = (dynamic_vs_heuristic_df["total_distance"] - dynamic_vs_heuristic_df["heuristic_total_distance"]) / dynamic_vs_heuristic_df["heuristic_total_distance"] * 100.0
dynamic_vs_heuristic_df["vehicle_difference_vs_heuristic"] = dynamic_vs_heuristic_df["num_vehicles"] - dynamic_vs_heuristic_df["heuristic_num_vehicles"]
dynamic_vs_heuristic_df["service_level_difference_vs_heuristic"] = dynamic_vs_heuristic_df["service_level"] - dynamic_vs_heuristic_df["heuristic_service_level"]
dynamic_vs_heuristic_df["rejection_difference_vs_heuristic"] = dynamic_vs_heuristic_df["rejection_rate"] - dynamic_vs_heuristic_df["heuristic_rejection_rate"]
dynamic_vs_heuristic_df["runtime_ratio_vs_heuristic"] = dynamic_vs_heuristic_df["computational_time"] / dynamic_vs_heuristic_df["heuristic_computational_time"].replace({0: np.nan})
dynamic_vs_heuristic_df["win_vs_heuristic"] = dynamic_vs_heuristic_df["total_cost"] < dynamic_vs_heuristic_df["heuristic_total_cost"]

print_table("dynamic_vs_heuristic_df", dynamic_vs_heuristic_df)


dynamic_vs_heuristic_df: 15187 rows x 71 columns


,run_id,scenario_id,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,seed,model_type,model_name,strategy,model_family,model_customer_size,allow_rejection,feasible,error_message,num_vehicles,total_distance,total_cost,customers_served,customers_rejected,service_level,rejection_rate,average_lateness,computational_time,routes,timestamp,solomon_class,solomon_type,lateness_proxy,cost_per_customer,distance_per_customer,customers_per_vehicle,method_label,configuration_id,num_reoptimization_events,runtime_per_customer,runtime_per_event,total_lateness,lateness_count,max_lateness,mean_lateness,source_file,dynamic_customer_ids,reveal_times,feasible_scenario,dropped_reason,solomon_class_scenario,solomon_type_scenario,num_dynamic_customers,num_static_customers,min_reveal_time,mean_reveal_time,max_reveal_time,reveal_time_std,mean_reveal_fraction,max_reveal_fraction,source_file_scenario,heuristic_total_cost,heuristic_total_distance,heuristic_num_vehicles,heuristic_service_level,heuristic_rejection_rate,heuristic_computational_time,heuristic_configuration_id,gap_vs_heuristic_%,distance_gap_vs_heuristic_%,vehicle_difference_vs_heuristic,service_level_difference_vs_heuristic,rejection_difference_vs_heuristic,runtime_ratio_vs_heuristic,win_vs_heuristic
0,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,ai,general_50,greedy,general,50,False,False,None,7,495.0,1195.0,50,0,1.0,0.0,0.66,0.0,"[{'node_ids': [13, 17, 18, 19, 15, 16, 14, 12,...",2026-05-02 14:54:23.464962,C,1,0.66,23.90,9.90,7.142857,AI | greedy | general-50,ai|greedy|general|50|general_50|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,1027.0,427.0,6,1.0,0.0,0.0,heuristic|NA|NA|NA|ortools|reject-0,16.358325,15.925059,1,0.0,0.0,NaN,False
1,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,ai,general_50,greedy,general,50,False,False,None,6,735.0,1335.0,50,0,1.0,0.0,29.74,0.0,"[{'node_ids': [5, 31, 35, 37, 38, 39, 36, 34, ...",2026-05-01 18:22:44.016495,C,1,29.74,26.70,14.70,8.333333,AI | greedy | general-50,ai|greedy|general|50|general_50|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,1027.0,427.0,6,1.0,0.0,0.0,heuristic|NA|NA|NA|ortools|reject-0,29.990263,72.131148,0,0.0,0.0,NaN,False
2,0,C101_n50_seed3765530_dod0p3_cut0p5,C101.txt,50,0.3,0.5,3765530,ai,general_50,greedy,general,50,False,False,None,7,757.0,1457.0,50,0,1.0,0.0,23.94,0.0,"[{'node_ids': [13, 17, 18, 19, 15, 10, 16, 14,...",2026-05-02 14:54:27.321369,C,1,23.94,29.14,15.14,7.142857,AI | greedy | general-50,ai|greedy|general|50|general_50|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[547.6407110584447, 24.3843598096617, 270.6860...",True,None,C,1,15,35.0,11.001816,169.944562,547.640711,178.821095,0.150794,0.485928,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,1026.0,426.0,6,1.0,0.0,0.0,heuristic|NA|NA|NA|ortools|reject-0,42.007797,77.699531,1,0.0,0.0,NaN,False
3,0,C101_n50_seed3765530_dod0p3_cut0p5,C101.txt,50,0.3,0.5,3765530,ai,general_50,greedy,general,50,False,False,None,6,823.0,1423.0,50,0,1.0,0.0,46.24,0.0,"[{'node_ids': [5, 3, 10, 11, 9, 6, 4, 2, 1], '...",2026-05-01 18:22:59.569528,C,1,46.24,28.46,16.46,8.333333,AI | greedy | general-50,ai|greedy|general|50|general_50|reject-0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[547.6407110584447, 24.3843598096617, 2

## 11. Overall dynamic method comparison


In [11]:
overall_dynamic_comparison = dynamic_vs_heuristic_df.groupby(
    ["model_type", "strategy", "model_family", "model_customer_size", "model_name"],
    dropna=False,
    as_index=False,
).agg(
    mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
    median_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "median"),
    std_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "std"),
    win_rate_vs_heuristic=("win_vs_heuristic", "mean"),
    mean_total_cost=("total_cost", "mean"),
    mean_total_distance=("total_distance", "mean"),
    mean_num_vehicles=("num_vehicles", "mean"),
    mean_service_level=("service_level", "mean"),
    mean_rejection_rate=("rejection_rate", "mean"),
    mean_average_lateness=("average_lateness", "mean"),
    mean_computational_time=("computational_time", "mean"),
)
best_configurations_overall = overall_dynamic_comparison.sort_values(
    ["mean_gap_vs_heuristic_pct", "mean_rejection_rate", "mean_average_lateness", "mean_computational_time"],
    kind="stable",
).head(10).reset_index(drop=True)

print_table("overall_dynamic_comparison", overall_dynamic_comparison)
print_table("best_configurations_overall", best_configurations_overall)

fig, _ = boxplot_or_placeholder(dynamic_vs_heuristic_df, "strategy", "gap_vs_heuristic_%", "Gap vs heuristic by model type and strategy", hue="model_type")
save_figure(fig, "dynamic_comparison", "gap_vs_heuristic_boxplot.png")

win_rate_bar = overall_dynamic_comparison.copy()
fig, _ = barplot_or_placeholder(win_rate_bar, "model_name", "win_rate_vs_heuristic", "Win rate vs heuristic by configuration", hue="model_type")
save_figure(fig, "dynamic_comparison", "win_rate_bar.png")

fig, _ = scatter_or_placeholder(dynamic_vs_heuristic_df, "computational_time", "gap_vs_heuristic_%", "Runtime vs gap vs heuristic", hue="model_type")
save_figure(fig, "dynamic_comparison", "runtime_gap_scatter.png")


overall_dynamic_comparison: 14 rows x 16 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_total_cost,mean_total_distance,mean_num_vehicles,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,ai,NA,general,50,general_50,126.532610,90.794702,115.930689,0.010275,3418.415976,2465.681472,9.527345,0.999850,0.000150,32.431141,0.0
1,ai,NA,lateness,50,routefinder_with_lateness_50,104.985369,90.576314,79.949723,0.010603,3192.469848,2309.500994,8.829689,0.999870,0.000130,32.087865,0.0
2,ai,NA,lateness,75,routefinder_with_lateness_75,103.640733,87.693654,73.411643,0.010272,3273.013917,2328.182903,9.448310,0.999576,0.000424,31.965394,0.0
3,ai,NA,solomon,50,routefinder_solomon_generated_50,145.866734,127.370031,91.745345,0.000000,3881.751740,2773.465363,11.082864,0.996053,0.003947,22.095763,0.0
4,ai,NA,solomon,75,routefinder_solomon_generated_75,194.933280,162.356322,161.470591,0.000331,4341.901558,3164.970832,11.769307,0.999985,0.000015,18.005265,0.0
5,ai,greedy,general,50,general_50,27.708822,26.116804,22.032701,0.100000,1388.200000,723.200000,6.650000,1.000000,0.000000,28.810000,0.0
6,ai,greedy,lateness,50,routefinder_with_lateness_50,45.285443,30.087634,53.251964,0.157895,1539.421053,828.894737,7.105263,1.000000,0.000000,31.232632,0.0
7,ai,greedy,lateness,75,routefinder_with_lateness_75,21.389464,22.348195,22.728507,0.222222,1296.333333,690.777778,6.055556,1.000000,0.000000,38.221111,0.0
8,ai,greedy,solomon,50,routefinder_solomon_generated_50,221.801998,194.017178,75.754699,0.000000,3466.650000,2246.650000,12.200000,1.000000,0.000000,10.812000,0.0
9,ai,greedy,solomon,75,routefinder_solomon_generated_75,335.871431,327.077224,77.686156,0.000000,4635.473684,2935.473684,17.000000,1.000000,0.000000,4.070526,0.0


best_configurations_overall: 10 rows x 16 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_total_cost,mean_total_distance,mean_num_vehicles,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,hybrid,greedy,general,50,hybrid:general_50,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
1,hybrid,greedy,lateness,50,hybrid:routefinder_with_lateness_50,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
2,hybrid,greedy,solomon,50,hybrid:routefinder_solomon_generated_50,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
3,hybrid,greedy,solomon,75,hybrid:routefinder_solomon_generated_75,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
4,ai,greedy,lateness,75,routefinder_with_lateness_75,21.389464,22.348195,22.728507,0.222222,1296.333333,690.777778,6.055556,1.000000,0.000000,38.221111,0.0
5,ai,greedy,general,50,general_50,27.708822,26.116804,22.032701,0.100000,1388.200000,723.200000,6.650000,1.000000,0.000000,28.810000,0.0
6,ai,greedy,lateness,50,routefinder_with_lateness_50,45.285443,30.087634,53.251964,0.157895,1539.421053,828.894737,7.105263,1.000000,0.000000,31.232632,0.0
7,ai,NA,lateness,75,routefinder_with_lateness_75,103.640733,87.693654,73.411643,0.010272,3273.013917,2328.182903,9.448310,0.999576,0.000424,31.965394,0.0
8,ai,NA,lateness,50,routefinder_with_lateness_50,104.985369,90.576314,79.949723,0.010603,3192.469848,2309.500994,8.829689,0.999870,0.000130,32.087865,0.0
9,ai,NA,general,50,general_50,126.532610,90.794702,115.930689,0.010275,3418.415976,2465.681472,9.527345,0.999850,0.000150,32.431141,0.0


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/dynamic_comparison/runtime_gap_scatter.png')

## 12. Hybrid vs AI comparison


In [12]:
ai_df = dynamic_eval_df[dynamic_eval_df["model_type"].eq("ai")].copy()
hybrid_df = dynamic_eval_df[dynamic_eval_df["model_type"].eq("hybrid")].copy()

hybrid_vs_ai_df = hybrid_df.merge(
    ai_df[
        [
            "scenario_id",
            "strategy",
            "model_family",
            "model_customer_size",
            "total_cost",
            "total_distance",
            "num_vehicles",
            "service_level",
            "rejection_rate",
            "computational_time",
        ]
    ],
    on=["scenario_id", "strategy", "model_family", "model_customer_size"],
    how="inner",
    suffixes=("_hybrid", "_ai"),
)
if hybrid_vs_ai_df.empty:
    hybrid_vs_ai_df = pd.DataFrame(
        columns=[
            "scenario_id",
            "strategy",
            "model_family",
            "model_customer_size",
            "hybrid_improvement_over_ai_%",
            "hybrid_distance_improvement_%",
            "hybrid_vehicle_difference",
            "hybrid_service_level_difference",
            "hybrid_rejection_difference",
            "hybrid_runtime_overhead_%",
            "hybrid_win_over_ai",
        ]
    )
else:
    hybrid_vs_ai_df["hybrid_improvement_over_ai_%"] = (hybrid_vs_ai_df["total_cost_ai"] - hybrid_vs_ai_df["total_cost_hybrid"]) / hybrid_vs_ai_df["total_cost_ai"] * 100.0
    hybrid_vs_ai_df["hybrid_distance_improvement_%"] = (hybrid_vs_ai_df["total_distance_ai"] - hybrid_vs_ai_df["total_distance_hybrid"]) / hybrid_vs_ai_df["total_distance_ai"] * 100.0
    hybrid_vs_ai_df["hybrid_vehicle_difference"] = hybrid_vs_ai_df["num_vehicles_hybrid"] - hybrid_vs_ai_df["num_vehicles_ai"]
    hybrid_vs_ai_df["hybrid_service_level_difference"] = hybrid_vs_ai_df["service_level_hybrid"] - hybrid_vs_ai_df["service_level_ai"]
    hybrid_vs_ai_df["hybrid_rejection_difference"] = hybrid_vs_ai_df["rejection_rate_hybrid"] - hybrid_vs_ai_df["rejection_rate_ai"]
    hybrid_vs_ai_df["hybrid_runtime_overhead_%"] = (hybrid_vs_ai_df["computational_time_hybrid"] - hybrid_vs_ai_df["computational_time_ai"]) / hybrid_vs_ai_df["computational_time_ai"].replace({0: np.nan}) * 100.0
    hybrid_vs_ai_df["hybrid_win_over_ai"] = hybrid_vs_ai_df["total_cost_hybrid"] < hybrid_vs_ai_df["total_cost_ai"]

hybrid_improvement_by_strategy = aggregate_metrics(hybrid_vs_ai_df, ["strategy"], ["hybrid_improvement_over_ai_%", "hybrid_runtime_overhead_%"])
hybrid_improvement_by_model = aggregate_metrics(hybrid_vs_ai_df, ["model_family", "model_customer_size"], ["hybrid_improvement_over_ai_%", "hybrid_runtime_overhead_%"])
hybrid_improvement_by_dynamic_setting = aggregate_metrics(hybrid_vs_ai_df, ["degree_of_dynamicity", "cutoff_time"] if "degree_of_dynamicity" in hybrid_vs_ai_df.columns else ["strategy"], ["hybrid_improvement_over_ai_%"])

print_table("hybrid_vs_ai_df", hybrid_vs_ai_df)
print_table("hybrid_improvement_by_strategy", hybrid_improvement_by_strategy)
print_table("hybrid_improvement_by_model", hybrid_improvement_by_model)
print_table("hybrid_improvement_by_dynamic_setting", hybrid_improvement_by_dynamic_setting)

fig, _ = boxplot_or_placeholder(hybrid_vs_ai_df, "strategy", "hybrid_improvement_over_ai_%", "Hybrid improvement over AI by strategy")
save_figure(fig, "hybrid_vs_ai", "hybrid_improvement_boxplot.png")

fig, _ = scatter_or_placeholder(hybrid_vs_ai_df, "total_cost_ai", "total_cost_hybrid", "AI total cost vs Hybrid total cost", hue="strategy")
save_figure(fig, "hybrid_vs_ai", "ai_vs_hybrid_scatter.png")

if {"degree_of_dynamicity", "cutoff_time", "hybrid_improvement_over_ai_%"}.issubset(hybrid_vs_ai_df.columns):
    heatmap_df = hybrid_vs_ai_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(value=("hybrid_improvement_over_ai_%", "mean"))
    fig, _ = scenario_heatmap(heatmap_df, "value", "Hybrid improvement by DoD x cutoff", cmap="coolwarm")
else:
    fig, _ = placeholder_figure("Hybrid improvement by DoD x cutoff", "Hybrid rows are missing or do not carry dynamic setting columns.")
save_figure(fig, "hybrid_vs_ai", "hybrid_heatmap.png")

win_rate_hybrid = hybrid_vs_ai_df.groupby(["model_family", "model_customer_size"], as_index=False).agg(win_rate=("hybrid_win_over_ai", "mean"))
fig, _ = barplot_or_placeholder(win_rate_hybrid, "model_family", "win_rate", "Hybrid win rate over AI by model", hue="model_customer_size")
save_figure(fig, "hybrid_vs_ai", "hybrid_win_rate_bar.png")


hybrid_vs_ai_df: 8 rows x 70 columns


,run_id,scenario_id,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,seed,model_type,model_name,strategy,model_family,model_customer_size,allow_rejection,feasible,error_message,num_vehicles_hybrid,total_distance_hybrid,total_cost_hybrid,customers_served,customers_rejected,service_level_hybrid,rejection_rate_hybrid,average_lateness,computational_time_hybrid,routes,timestamp,solomon_class,solomon_type,lateness_proxy,cost_per_customer,distance_per_customer,customers_per_vehicle,method_label,configuration_id,num_reoptimization_events,runtime_per_customer,runtime_per_event,total_lateness,lateness_count,max_lateness,mean_lateness,source_file,dynamic_customer_ids,reveal_times,feasible_scenario,dropped_reason,solomon_class_scenario,solomon_type_scenario,num_dynamic_customers,num_static_customers,min_reveal_time,mean_reveal_time,max_reveal_time,reveal_time_std,mean_reveal_fraction,max_reveal_fraction,source_file_scenario,total_cost_ai,total_distance_ai,num_vehicles_ai,service_level_ai,rejection_rate_ai,computational_time_ai,hybrid_improvement_over_ai_%,hybrid_distance_improvement_%,hybrid_vehicle_difference,hybrid_service_level_difference,hybrid_rejection_difference,hybrid_runtime_overhead_%,hybrid_win_over_ai
0,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,hybrid,hybrid:general_50,greedy,general,50,False,True,None,6,427.0,1027.0,50,0,1.0,0.0,0.0,0.0,"[{'node_ids': [5, 3, 7, 8, 10, 11, 9, 6, 4, 2,...",2026-05-02 19:36:35.654618,C,1,0.0,20.54,8.54,8.333333,HYBRID | greedy | general-50,hybrid|greedy|general|50|hybrid:general_50|rej...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,1195.0,495.0,7,1.0,0.0,0.0,14.058577,13.737374,-1,0.0,0.0,NaN,True
1,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,hybrid,hybrid:general_50,greedy,general,50,False,True,None,6,427.0,1027.0,50,0,1.0,0.0,0.0,0.0,"[{'node_ids': [5, 3, 7, 8, 10, 11, 9, 6, 4, 2,...",2026-05-02 19:36:35.654618,C,1,0.0,20.54,8.54,8.333333,HYBRID | greedy | general-50,hybrid|greedy|general|50|hybrid:general_50|rej...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,1335.0,735.0,6,1.0,0.0,0.0,23.071161,41.904762,0,0.0,0.0,NaN,True
2,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,hybrid,hybrid:routefinder_solomon_generated_50,greedy,solomon,50,False,True,None,6,427.0,1027.0,50,0,1.0,0.0,0.0,0.0,"[{'node_ids': [5, 3, 7, 8, 10, 11, 9, 6, 4, 2,...",2026-05-02 19:36:52.431358,C,1,0.0,20.54,8.54,8.333333,HYBRID | greedy | solomon-50,hybrid|greedy|solomon|50|hybrid:routefinder_so...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,C,1,15,35.0,7.688283,82.228586,219.056284,71.301003,0.072962,0.194371,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,3782.0,2182.0,16,1.0,0.0,0.0,72.845056,80.430797,-10,0.0,0.0,NaN,True
3,0,C101_n50_seed3765530_dod0p3_cut0p2,C101.txt,50,0.3,0.2,3765530,hybrid,hybrid:routefinder_solomon_generated_50,greedy,solomon,50,False,True,None,6,427.0,1027.0,50,0,1.0,0.0,0.0,0.0,"[{'node_ids': [5, 3, 7, 8, 10, 11, 9, 6, 4, 2,...",2026-05-02 19:36:52.431358,C,1,0.0,20.54,8.54,8.333333,HYBRID | greedy | solomon-50,hybrid|greedy|solomon|50|hybrid:routefinder_so...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,/Users/giuseppe/Documents/personal/fyp-vrp/dvr...,"[2, 3, 4, 9, 12, 24, 25, 28, 29, 33, 35, 44, 4...","[219.0562844233779, 24.3843598096617, 108.2744...",True,None,

hybrid_improvement_by_strategy: 1 rows x 8 columns


,strategy,mean_hybrid_improvement_over_ai_%,median_hybrid_improvement_over_ai_%,std_hybrid_improvement_over_ai_%,mean_hybrid_runtime_overhead_%,median_hybrid_runtime_overhead_%,std_hybrid_runtime_overhead_%,rows
0,greedy,41.925513,36.85311,31.294586,NaN,NaN,NaN,8


hybrid_improvement_by_model: 4 rows x 9 columns


,model_family,model_customer_size,mean_hybrid_improvement_over_ai_%,median_hybrid_improvement_over_ai_%,std_hybrid_improvement_over_ai_%,mean_hybrid_runtime_overhead_%,median_hybrid_runtime_overhead_%,std_hybrid_runtime_overhead_%,rows
0,general,50,18.564869,18.564869,6.372859,NaN,NaN,NaN,2
1,lateness,50,10.672399,10.672399,17.615930,NaN,NaN,NaN,2
2,solomon,50,61.711267,61.711267,15.745555,NaN,NaN,NaN,2
3,solomon,75,76.753516,76.753516,1.647857,NaN,NaN,NaN,2


hybrid_improvement_by_dynamic_setting: 1 rows x 6 columns


,degree_of_dynamicity,cutoff_time,mean_hybrid_improvement_over_ai_%,median_hybrid_improvement_over_ai_%,std_hybrid_improvement_over_ai_%,rows
0,0.3,0.2,41.925513,36.85311,31.294586,8


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/hybrid_vs_ai/hybrid_win_rate_bar.png')

## 13. Strategy comparison


In [13]:
strategy_df = dynamic_vs_heuristic_df[dynamic_vs_heuristic_df["strategy"].isin(["greedy", "sampling", "multistart"])].copy()
strategy_summary = strategy_df.groupby(["model_type", "strategy"], dropna=False, as_index=False).agg(
    mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
    median_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "median"),
    win_rate_vs_heuristic=("win_vs_heuristic", "mean"),
    mean_computational_time=("computational_time", "mean"),
    mean_service_level=("service_level", "mean"),
    mean_rejection_rate=("rejection_rate", "mean"),
)
strategy_average_ranks = average_rank_per_scenario(strategy_df, "gap_vs_heuristic_%", ["model_type", "strategy"], ascending=True)
strategy_summary_ai = strategy_summary[strategy_summary["model_type"].eq("ai")].reset_index(drop=True)
strategy_summary_hybrid = strategy_summary[strategy_summary["model_type"].eq("hybrid")].reset_index(drop=True)

print_table("strategy_summary_ai", strategy_summary_ai)
print_table("strategy_summary_hybrid", strategy_summary_hybrid)
print_table("strategy_average_ranks", strategy_average_ranks)

fig, _ = boxplot_or_placeholder(strategy_df, "strategy", "gap_vs_heuristic_%", "Gap vs heuristic by strategy", hue="model_type")
save_figure(fig, "strategy", "strategy_gap_boxplot.png")

fig, _ = scatter_or_placeholder(strategy_df, "computational_time", "gap_vs_heuristic_%", "Runtime vs gap by strategy", hue="strategy")
save_figure(fig, "strategy", "strategy_runtime_gap_scatter.png")

rank_bar = strategy_average_ranks.copy()
fig, _ = barplot_or_placeholder(rank_bar, "strategy", "average_rank_per_scenario", "Average rank by strategy", hue="model_type")
save_figure(fig, "strategy", "strategy_average_rank_bar.png")


strategy_summary_ai: 1 rows x 8 columns


,model_type,strategy,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_computational_time,mean_service_level,mean_rejection_rate
0,ai,greedy,131.429243,52.816964,0.09375,0.0,1.0,0.0


strategy_summary_hybrid: 1 rows x 8 columns


,model_type,strategy,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_computational_time,mean_service_level,mean_rejection_rate
0,hybrid,greedy,0.0,0.0,0.0,0.0,1.0,0.0


strategy_average_ranks: 2 rows x 3 columns


,model_type,strategy,average_rank_per_scenario
0,ai,greedy,4.572917
1,hybrid,greedy,3.500000


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/strategy/strategy_average_rank_bar.png')

## 14. Model family comparison


In [14]:
dynamic_vs_heuristic_df["trained_size_matches_eval_size"] = dynamic_vs_heuristic_df["model_customer_size"].astype(str) == dynamic_vs_heuristic_df["evaluation_size"].astype(str)
model_family_summary = dynamic_vs_heuristic_df.groupby(
    ["model_type", "model_family", "model_customer_size", "evaluation_size"],
    dropna=False,
    as_index=False,
).agg(
    mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
    median_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "median"),
    win_rate_vs_heuristic=("win_vs_heuristic", "mean"),
    mean_service_level=("service_level", "mean"),
    mean_rejection_rate=("rejection_rate", "mean"),
    mean_average_lateness=("average_lateness", "mean"),
    mean_computational_time=("computational_time", "mean"),
)
trained_size_transfer_summary = dynamic_vs_heuristic_df.groupby(
    ["model_type", "model_family", "model_customer_size", "evaluation_size", "trained_size_matches_eval_size"],
    dropna=False,
    as_index=False,
).agg(
    mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
    win_rate_vs_heuristic=("win_vs_heuristic", "mean"),
)

print_table("model_family_summary", model_family_summary)
print_table("trained_size_transfer_summary", trained_size_transfer_summary)

fig, _ = boxplot_or_placeholder(dynamic_vs_heuristic_df, "model_family", "gap_vs_heuristic_%", "Gap vs heuristic by model family", hue="model_type")
save_figure(fig, "model_family", "model_family_gap_boxplot.png")

heatmap_df = model_family_summary.groupby(["model_customer_size", "evaluation_size"], as_index=False).agg(value=("mean_gap_vs_heuristic_pct", "mean"))
fig, _ = scenario_heatmap(heatmap_df.rename(columns={"model_customer_size": "degree_of_dynamicity", "evaluation_size": "cutoff_time"}), "value", "Model customer size x evaluation size", cmap="YlGnBu")
save_figure(fig, "model_family", "size_transfer_heatmap.png")

model_name_bar = dynamic_vs_heuristic_df.groupby("model_name", as_index=False).agg(mean_gap=("gap_vs_heuristic_%", "mean"))
fig, _ = barplot_or_placeholder(model_name_bar, "model_name", "mean_gap", "Mean gap by model name")
save_figure(fig, "model_family", "model_name_gap_bar.png")


model_family_summary: 14 rows x 11 columns


,model_type,model_family,model_customer_size,evaluation_size,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,ai,general,50,50,113.196552,75.880551,0.015738,0.999921,0.000079,26.266428,0.0
1,ai,general,50,75,138.676136,101.772904,0.005952,0.999780,0.000220,38.600960,0.0
2,ai,lateness,50,50,96.744862,82.913547,0.015738,0.999882,0.000118,24.639169,0.0
3,ai,lateness,50,75,112.546528,97.186839,0.007275,0.999859,0.000141,39.589858,0.0
4,ai,lateness,75,50,96.888772,82.130896,0.015092,0.999921,0.000079,25.084589,0.0
5,ai,lateness,75,75,109.467099,92.076486,0.007937,0.999233,0.000767,38.975281,0.0
6,ai,solomon,50,50,131.996986,120.217835,0.000000,0.999895,0.000105,18.388826,0.0
7,ai,solomon,50,75,160.860168,136.224350,0.000000,0.992231,0.007769,25.685316,0.0
8,ai,solomon,75,50,202.638941,170.826273,0.000000,0.999987,0.000013,14.570111,0.0
9,ai,solomon,75,75,188.937511,155.515955,0.000661,0.999982,0.000018,21.292577,0.0


trained_size_transfer_summary: 14 rows x 7 columns


,model_type,model_family,model_customer_size,evaluation_size,trained_size_matches_eval_size,mean_gap_vs_heuristic_pct,win_rate_vs_heuristic
0,ai,general,50,50,True,113.196552,0.015738
1,ai,general,50,75,False,138.676136,0.005952
2,ai,lateness,50,50,True,96.744862,0.015738
3,ai,lateness,50,75,False,112.546528,0.007275
4,ai,lateness,75,50,False,96.888772,0.015092
5,ai,lateness,75,75,True,109.467099,0.007937
6,ai,solomon,50,50,True,131.996986,0.000000
7,ai,solomon,50,75,False,160.860168,0.000000
8,ai,solomon,75,50,False,202.638941,0.000000
9,ai,solomon,75,75,True,188.937511,0.000661


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/model_family/model_name_gap_bar.png')

## 15. Dynamic sensitivity analysis


In [15]:
sensitivity_by_DoD = aggregate_metrics(dynamic_vs_heuristic_df, ["degree_of_dynamicity", "model_type", "strategy", "model_family", "model_customer_size"], ["gap_vs_heuristic_%", "service_level", "rejection_rate", "average_lateness", "computational_time"])
sensitivity_by_cutoff = aggregate_metrics(dynamic_vs_heuristic_df, ["cutoff_time", "model_type", "strategy", "model_family", "model_customer_size"], ["gap_vs_heuristic_%", "service_level", "rejection_rate", "average_lateness", "computational_time"])
sensitivity_by_DoD_cutoff = aggregate_metrics(dynamic_vs_heuristic_df, ["degree_of_dynamicity", "cutoff_time", "model_type", "strategy", "model_family", "model_customer_size"], ["gap_vs_heuristic_%", "service_level", "rejection_rate", "average_lateness", "computational_time"])
best_config_by_DoD_cutoff = (
    sensitivity_by_DoD_cutoff.sort_values(["degree_of_dynamicity", "cutoff_time", "mean_gap_vs_heuristic_%"], kind="stable")
    if "mean_gap_vs_heuristic_%" in sensitivity_by_DoD_cutoff.columns
    else pd.DataFrame()
)

print_table("sensitivity_by_DoD", sensitivity_by_DoD)
print_table("sensitivity_by_cutoff", sensitivity_by_cutoff)
print_table("sensitivity_by_DoD_cutoff", sensitivity_by_DoD_cutoff)

heatmap_gap = dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(value=("gap_vs_heuristic_%", "mean"))
fig, _ = scenario_heatmap(heatmap_gap, "value", "Mean gap vs heuristic by DoD x cutoff", cmap="coolwarm")
save_figure(fig, "sensitivity", "gap_heatmap.png")

heatmap_rej = dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(value=("rejection_rate", "mean"))
fig, _ = scenario_heatmap(heatmap_rej, "value", "Rejection rate by DoD x cutoff", cmap="magma")
save_figure(fig, "sensitivity", "rejection_heatmap.png")

heatmap_service = dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(value=("service_level", "mean"))
fig, _ = scenario_heatmap(heatmap_service, "value", "Service level by DoD x cutoff", cmap="viridis")
save_figure(fig, "sensitivity", "service_heatmap.png")

line_df = dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(gap=("gap_vs_heuristic_%", "mean"))
fig, _ = lineplot_or_placeholder(line_df, "degree_of_dynamicity", "gap", "Gap vs heuristic by DoD separated by cutoff", hue="cutoff_time")
save_figure(fig, "sensitivity", "gap_lineplot.png")


sensitivity_by_DoD: 34 rows x 21 columns


,degree_of_dynamicity,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_average_lateness,median_average_lateness,std_average_lateness,mean_computational_time,median_computational_time,std_computational_time,rows
0,0.3,ai,NA,general,50,137.720173,95.780399,124.134720,0.999841,1.0,0.001827,0.000159,0.0,0.001827,24.909539,9.226667,31.140544,0.0,0.0,0.0,1005
1,0.3,ai,NA,lateness,50,109.995796,96.728138,80.820401,0.999741,1.0,0.002707,0.000259,0.0,0.002707,24.688849,9.080000,31.900475,0.0,0.0,0.0,1005
2,0.3,ai,NA,lateness,75,109.597661,89.675090,80.159135,0.999416,1.0,0.006634,0.000584,0.0,0.006634,24.642793,9.540000,30.982015,0.0,0.0,0.0,1005
3,0.3,ai,NA,solomon,50,147.432282,130.320513,90.781195,0.995914,1.0,0.026879,0.004086,0.0,0.026879,17.175191,6.320000,20.938008,0.0,0.0,0.0,1005
4,0.3,ai,NA,solomon,75,186.980795,158.756684,143.721512,0.999967,1.0,0.000758,0.000033,0.0,0.000758,13.755590,4.680000,17.989725,0.0,0.0,0.0,1005
5,0.3,ai,greedy,general,50,41.976558,40.860215,13.800203,1.000000,1.0,0.000000,0.000000,0.0,0.000000,25.260000,23.940000,20.571988,0.0,0.0,0.0,9
6,0.3,ai,greedy,lateness,50,62.566091,43.108504,55.727009,1.000000,1.0,0.000000,0.000000,0.0,0.000000,16.673333,7.660000,18.394760,0.0,0.0,0.0,9
7,0.3,ai,greedy,lateness,75,35.634857,33.918129,19.253265,1.000000,1.0,0.000000,0.000000,0.0,0.000000,33.682222,29.280000,14.938395,0.0,0.0,0.0,9
8,0.3,ai,greedy,solomon,50,250.475211,271.749756,95.341558,1.000000,1.0,0.000000,0.000000,0.0,0.000000,8.675556,6.660000,5.264454,0.0,0.0,0.0,9
9,0.3,ai,greedy,solomon,75,364.333485,352.872444,60.607220,1.000000,1.0,0.000000,0.000000,0.0,0.000000,1.793333,1.660000,1.492917,0.0,0.0,0.0,9


sensitivity_by_cutoff: 34 rows x 21 columns


,cutoff_time,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_average_lateness,median_average_lateness,std_average_lateness,mean_computational_time,median_computational_time,std_computational_time,rows
0,0.2,ai,NA,general,50,117.859549,75.880551,120.969962,0.999748,1.0,0.003917,0.000252,0.0,0.003917,7.818452,3.666667,12.387521,0.0,0.0,0.0,1005
1,0.2,ai,NA,lateness,50,84.959262,75.293480,68.116582,1.000000,1.0,0.000000,0.000000,0.0,0.000000,7.718244,3.616667,12.674219,0.0,0.0,0.0,1006
2,0.2,ai,NA,lateness,75,87.558966,69.725142,72.782179,0.999298,1.0,0.006990,0.000702,0.0,0.006990,7.517598,3.633333,11.971812,0.0,0.0,0.0,1006
3,0.2,ai,NA,solomon,50,121.517676,104.438964,87.343550,0.991973,1.0,0.038306,0.008027,0.0,0.038306,4.676412,2.600000,7.015617,0.0,0.0,0.0,1005
4,0.2,ai,NA,solomon,75,154.635147,116.983122,139.633487,0.999987,1.0,0.000421,0.000013,0.0,0.000421,4.127441,2.613333,5.709277,0.0,0.0,0.0,1005
5,0.2,ai,greedy,general,50,15.774837,15.546640,16.397121,1.000000,1.0,0.000000,0.000000,0.0,0.000000,19.865714,9.580000,20.328550,0.0,0.0,0.0,7
6,0.2,ai,greedy,lateness,50,7.802131,6.111536,15.274687,1.000000,1.0,0.000000,0.000000,0.0,0.000000,31.702857,38.680000,23.135256,0.0,0.0,0.0,7
7,0.2,ai,greedy,lateness,75,14.433095,12.938816,16.662284,1.000000,1.0,0.000000,0.000000,0.0,0.000000,41.794286,29.280000,25.254768,0.0,0.0,0.0,7
8,0.2,ai,greedy,solomon,50,208.769097,194.270435,74.689247,1.000000,1.0,0.000000,0.000000,0.0,0.000000,12.160000,7.940000,10.843413,0.0,0.0,0.0,7
9,0.2,ai,greedy,solomon,75,323.019571,318.355065,67.054381,1.000000,1.0,0.000000,0.000000,0.0,0.000000,3.628571,2.220000,3.346099,0.0,0.0,0.0,7


sensitivity_by_DoD_cutoff: 94 rows x 22 columns


,degree_of_dynamicity,cutoff_time,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_average_lateness,median_average_lateness,std_average_lateness,mean_computational_time,median_computational_time,std_computational_time,rows
0,0.3,0.2,ai,NA,general,50,129.311191,81.244522,129.589370,0.999960,1.0,0.000728,0.000040,0.0,0.000728,6.474924,3.00,10.138115,0.0,0.0,0.0,335
1,0.3,0.2,ai,NA,lateness,50,91.639475,84.970817,68.953336,1.000000,1.0,0.000000,0.000000,0.0,0.000000,6.185393,2.84,10.239651,0.0,0.0,0.0,335
2,0.3,0.2,ai,NA,lateness,75,93.866556,75.505407,76.063005,0.999244,1.0,0.007219,0.000756,0.0,0.007219,6.217184,3.28,9.290146,0.0,0.0,0.0,335
3,0.3,0.2,ai,NA,solomon,50,126.691572,106.230366,87.838665,0.991403,1.0,0.041826,0.008597,0.0,0.041826,3.658785,2.10,5.388041,0.0,0.0,0.0,335
4,0.3,0.2,ai,NA,solomon,75,153.853185,119.241192,127.320098,1.000000,1.0,0.000000,0.000000,0.0,0.000000,3.545592,2.08,5.227492,0.0,0.0,0.0,335
5,0.3,0.2,ai,greedy,general,50,28.533642,29.990263,11.516304,1.000000,1.0,0.000000,0.000000,0.0,0.000000,12.113333,5.94,15.491744,0.0,0.0,0.0,3
6,0.3,0.2,ai,greedy,lateness,50,11.469908,6.074766,16.591625,1.000000,1.0,0.000000,0.000000,0.0,0.000000,16.153333,0.22,27.788086,0.0,0.0,0.0,3
7,0.3,0.2,ai,greedy,lateness,75,21.646806,29.503408,14.885476,1.000000,1.0,0.000000,0.000000,0.0,0.000000,31.546667,29.28,20.733136,0.0,0.0,0.0,3
8,0.3,0.2,ai,greedy,solomon,50,230.345963,268.257059,113.888571,1.000000,1.0,0.000000,0.000000,0.0,0.000000,9.546667,6.14,9.158217,0.0,0.0,0.0,3
9,0.3,0.2,ai,greedy,solomon,75,359.817141,352.872444,53.985819,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.766667,0.32,0.773649,0.0,0.0,0.0,3


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/sensitivity/gap_lineplot.png')

## 16. Seed robustness analysis


In [16]:
seed_variance_summary = dynamic_vs_heuristic_df.groupby(
    ["instance_id", "evaluation_size", "degree_of_dynamicity", "cutoff_time", "model_type", "strategy", "model_family", "model_customer_size"],
    dropna=False,
    as_index=False,
).agg(
    mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
    std_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "std"),
    min_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "min"),
    max_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "max"),
)
seed_variance_summary["worst_case_gap_vs_heuristic_%"] = seed_variance_summary["max_gap_vs_heuristic_pct"]
most_seed_sensitive_configurations = seed_variance_summary.sort_values("std_gap_vs_heuristic_pct", ascending=False, kind="stable").head(20).reset_index(drop=True)

print_table("seed_variance_summary", seed_variance_summary)
print_table("most_seed_sensitive_configurations", most_seed_sensitive_configurations)

fig, _ = boxplot_or_placeholder(dynamic_vs_heuristic_df, "seed", "gap_vs_heuristic_%", "Gap vs heuristic by seed", hue="model_type")
save_figure(fig, "sensitivity", "seed_gap_boxplot.png")

worst_case_bar = most_seed_sensitive_configurations.groupby(["model_type", "strategy"], as_index=False).agg(worst_case_gap=("worst_case_gap_vs_heuristic_%", "mean"))
fig, _ = barplot_or_placeholder(worst_case_bar, "strategy", "worst_case_gap", "Worst-case gap by configuration", hue="model_type")
save_figure(fig, "sensitivity", "worst_case_gap_bar.png")


seed_variance_summary: 5089 rows x 13 columns


,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,min_gap_vs_heuristic_pct,max_gap_vs_heuristic_pct,worst_case_gap_vs_heuristic_%
0,C101.txt,50,0.3,0.2,ai,NA,general,50,31.591853,1.251271,30.707071,32.476636,32.476636
1,C101.txt,50,0.3,0.2,ai,NA,lateness,50,26.222859,5.198834,22.546729,29.898990,29.898990
2,C101.txt,50,0.3,0.2,ai,NA,lateness,75,22.512626,14.159992,12.500000,32.525253,32.525253
3,C101.txt,50,0.3,0.2,ai,NA,solomon,50,160.268574,23.807197,143.434343,177.102804,177.102804
4,C101.txt,50,0.3,0.2,ai,NA,solomon,75,297.667681,25.271574,279.797980,315.537383,315.537383
5,C101.txt,50,0.3,0.2,ai,greedy,general,50,28.533642,11.516304,16.358325,39.252336,39.252336
6,C101.txt,50,0.3,0.2,ai,greedy,lateness,50,11.469908,16.591625,-1.752678,30.087634,30.087634
7,C101.txt,50,0.3,0.2,ai,greedy,lateness,75,21.646806,14.885476,4.479065,30.957944,30.957944
8,C101.txt,50,0.3,0.2,ai,greedy,solomon,50,230.345963,113.888571,102.336904,320.443925,320.443925
9,C101.txt,50,0.3,0.2,ai,greedy,solomon,75,359.817141,53.985819,309.639727,416.939252,416.939252


most_seed_sensitive_configurations: 20 rows x 13 columns


,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,min_gap_vs_heuristic_pct,max_gap_vs_heuristic_pct,worst_case_gap_vs_heuristic_%
0,C201.txt,75,0.5,0.5,ai,NA,lateness,50,438.451578,259.685487,178.431373,697.801047,697.801047
1,C208.txt,50,0.3,0.2,ai,NA,solomon,75,441.801684,230.718058,190.977444,644.971264,644.971264
2,C206.txt,50,0.7,0.2,ai,NA,solomon,75,519.424496,195.133797,347.591069,731.563845,731.563845
3,C205.txt,50,0.7,0.2,ai,NA,solomon,75,512.652184,193.780162,291.795482,654.185022,654.185022
4,C202.txt,50,0.7,0.8,ai,NA,solomon,75,655.197385,190.433611,539.812646,875.000000,875.000000
5,C202.txt,50,0.7,0.8,ai,NA,general,50,378.894538,177.098532,238.195991,577.761628,577.761628
6,C201.txt,50,0.3,0.5,ai,NA,solomon,75,606.903152,176.019364,464.530892,803.709428,803.709428
7,C201.txt,50,0.3,0.8,ai,NA,solomon,75,602.795312,169.506770,449.829352,785.030864,785.030864
8,C205.txt,75,0.7,0.8,ai,NA,solomon,75,643.177937,162.818267,514.229867,826.138614,826.138614
9,C202.txt,75,0.3,0.5,ai,NA,general,50,549.617580,149.849815,405.188679,704.355885,704.355885


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/sensitivity/worst_case_gap_bar.png')

## 17. Solomon structure analysis


In [17]:
results_by_solomon_class = aggregate_metrics(dynamic_vs_heuristic_df, ["solomon_class", "evaluation_size", "model_type", "strategy", "model_family", "model_customer_size"], ["gap_vs_heuristic_%", "service_level", "rejection_rate", "computational_time"])
results_by_solomon_type = aggregate_metrics(dynamic_vs_heuristic_df, ["solomon_type", "evaluation_size", "model_type", "strategy", "model_family", "model_customer_size"], ["gap_vs_heuristic_%", "service_level", "rejection_rate", "computational_time"])
results_by_customer_size = aggregate_metrics(dynamic_vs_heuristic_df, ["evaluation_size", "model_type", "strategy", "model_family", "model_customer_size"], ["gap_vs_heuristic_%", "service_level", "rejection_rate", "computational_time"])
best_config_by_solomon_structure = (
    results_by_solomon_class.sort_values(["solomon_class", "mean_gap_vs_heuristic_%"], kind="stable")
    if "mean_gap_vs_heuristic_%" in results_by_solomon_class.columns
    else pd.DataFrame()
)

print_table("results_by_solomon_class", results_by_solomon_class)
print_table("results_by_solomon_type", results_by_solomon_type)
print_table("results_by_customer_size", results_by_customer_size)

fig, _ = boxplot_or_placeholder(dynamic_vs_heuristic_df, "solomon_class", "gap_vs_heuristic_%", "Gap vs heuristic by Solomon class", hue="model_type")
save_figure(fig, "solomon_structure", "solomon_class_boxplot.png")

fig, _ = boxplot_or_placeholder(dynamic_vs_heuristic_df, "solomon_type", "gap_vs_heuristic_%", "Gap vs heuristic by Solomon type", hue="model_type")
save_figure(fig, "solomon_structure", "solomon_type_boxplot.png")

fig, _ = boxplot_or_placeholder(dynamic_vs_heuristic_df, "evaluation_size", "gap_vs_heuristic_%", "Gap vs heuristic by customer size", hue="model_type")
save_figure(fig, "solomon_structure", "customer_size_boxplot.png")

heatmap_solomon = dynamic_vs_heuristic_df.groupby(["solomon_class", "method_label"], as_index=False).agg(value=("gap_vs_heuristic_%", "mean"))
if heatmap_solomon.empty:
    fig, _ = placeholder_figure("Solomon class x method heatmap", "No data available.")
else:
    temp = heatmap_solomon.rename(columns={"solomon_class": "degree_of_dynamicity", "method_label": "cutoff_time"})
    fig, _ = scenario_heatmap(temp, "value", "Solomon class x method/configuration", cmap="coolwarm")
save_figure(fig, "solomon_structure", "solomon_method_heatmap.png")


results_by_solomon_class: 39 rows x 19 columns


,solomon_class,evaluation_size,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_computational_time,median_computational_time,std_computational_time,rows
0,C,50,ai,NA,general,50,177.535006,166.953486,142.980485,0.999956,1.0,0.000941,0.000044,0.0,0.000941,0.0,0.0,0.0,452
1,C,50,ai,NA,lateness,50,126.274154,114.655172,105.504740,0.999868,1.0,0.002099,0.000132,0.0,0.002099,0.0,0.0,0.0,453
2,C,50,ai,NA,lateness,75,112.040566,88.600289,98.684339,0.999868,1.0,0.001624,0.000132,0.0,0.001624,0.0,0.0,0.0,453
3,C,50,ai,NA,solomon,50,177.691378,153.946056,88.090676,0.999690,1.0,0.002808,0.000310,0.0,0.002808,0.0,0.0,0.0,452
4,C,50,ai,NA,solomon,75,366.380009,304.944617,187.328810,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,452
5,C,50,ai,greedy,general,50,27.708822,26.116804,22.032701,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
6,C,50,ai,greedy,lateness,50,45.285443,30.087634,53.251964,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19
7,C,50,ai,greedy,lateness,75,21.389464,22.348195,22.728507,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,18
8,C,50,ai,greedy,solomon,50,221.801998,194.017178,75.754699,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
9,C,50,ai,greedy,solomon,75,335.871431,327.077224,77.686156,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19


results_by_solomon_type: 29 rows x 19 columns


,solomon_type,evaluation_size,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_computational_time,median_computational_time,std_computational_time,rows
0,1,50,ai,NA,general,50,60.957945,43.925588,54.230866,0.999845,1.0,0.002026,0.000155,0.0,0.002026,0.0,0.0,0.0,776
1,1,50,ai,NA,lateness,50,56.570121,44.136598,43.721051,0.999768,1.0,0.002578,0.000232,0.0,0.002578,0.0,0.0,0.0,777
2,1,50,ai,NA,lateness,75,75.156725,68.620038,48.662936,0.999846,1.0,0.001752,0.000154,0.0,0.001752,0.0,0.0,0.0,777
3,1,50,ai,NA,solomon,50,91.286665,85.602053,36.919477,0.999794,1.0,0.002262,0.000206,0.0,0.002262,0.0,0.0,0.0,776
4,1,50,ai,NA,solomon,75,115.024575,79.899921,82.785379,0.999974,1.0,0.000718,0.000026,0.0,0.000718,0.0,0.0,0.0,776
5,1,50,ai,greedy,general,50,27.708822,26.116804,22.032701,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
6,1,50,ai,greedy,lateness,50,45.285443,30.087634,53.251964,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19
7,1,50,ai,greedy,lateness,75,21.389464,22.348195,22.728507,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,18
8,1,50,ai,greedy,solomon,50,221.801998,194.017178,75.754699,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
9,1,50,ai,greedy,solomon,75,335.871431,327.077224,77.686156,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19


results_by_customer_size: 19 rows x 18 columns


,evaluation_size,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_computational_time,median_computational_time,std_computational_time,rows
0,50,ai,NA,general,50,114.332601,77.098243,105.463723,0.999920,1.0,0.001456,0.000080,0.0,0.001456,0.0,0.0,0.0,1505
1,50,ai,NA,lateness,50,97.394085,83.057395,73.923518,0.999880,1.0,0.001855,0.000120,0.0,0.001855,0.0,0.0,0.0,1506
2,50,ai,NA,lateness,75,97.791154,82.579153,67.214796,0.999920,1.0,0.001260,0.000080,0.0,0.001260,0.0,0.0,0.0,1506
3,50,ai,NA,solomon,50,130.803563,119.056974,70.590194,0.999894,1.0,0.001627,0.000106,0.0,0.001627,0.0,0.0,0.0,1505
4,50,ai,NA,solomon,75,200.956936,169.657423,162.605590,0.999987,1.0,0.000516,0.000013,0.0,0.000516,0.0,0.0,0.0,1505
5,50,ai,greedy,general,50,27.708822,26.116804,22.032701,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
6,50,ai,greedy,lateness,50,45.285443,30.087634,53.251964,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19
7,50,ai,greedy,lateness,75,21.389464,22.348195,22.728507,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,18
8,50,ai,greedy,solomon,50,221.801998,194.017178,75.754699,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
9,50,ai,greedy,solomon,75,335.871431,327.077224,77.686156,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)
/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)
/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/solomon_structure/solomon_method_heatmap.png')

## 18. Static vs dynamic degradation


In [18]:
dynamic_mean_df = dynamic_eval_df.groupby(
    ["instance_id", "evaluation_size", "model_type", "strategy", "model_family", "model_customer_size", "model_name"],
    dropna=False,
    as_index=False,
).agg(
    mean_dynamic_total_cost=("total_cost", "mean"),
    mean_dynamic_total_distance=("total_distance", "mean"),
    mean_dynamic_num_vehicles=("num_vehicles", "mean"),
    mean_dynamic_service_level=("service_level", "mean"),
    mean_dynamic_rejection_rate=("rejection_rate", "mean"),
    mean_dynamic_runtime=("computational_time", "mean"),
)

static_vs_dynamic_df = dynamic_mean_df.merge(
    static_results_df,
    on=["instance_id", "evaluation_size", "model_type", "strategy", "model_family", "model_customer_size", "model_name"],
    how="left",
    suffixes=("_dynamic", "_static"),
)
static_vs_dynamic_df["dynamic_degradation_%"] = (static_vs_dynamic_df["mean_dynamic_total_cost"] - static_vs_dynamic_df["total_cost"]) / static_vs_dynamic_df["total_cost"] * 100.0
static_vs_dynamic_df["distance_degradation_%"] = (static_vs_dynamic_df["mean_dynamic_total_distance"] - static_vs_dynamic_df["total_distance"]) / static_vs_dynamic_df["total_distance"] * 100.0
static_vs_dynamic_df["vehicle_difference_dynamic_static"] = static_vs_dynamic_df["mean_dynamic_num_vehicles"] - static_vs_dynamic_df["num_vehicles"]
static_vs_dynamic_df["service_level_difference_dynamic_static"] = static_vs_dynamic_df["mean_dynamic_service_level"] - static_vs_dynamic_df["service_level"]
static_vs_dynamic_df["rejection_difference_dynamic_static"] = static_vs_dynamic_df["mean_dynamic_rejection_rate"] - static_vs_dynamic_df["rejection_rate"]
static_vs_dynamic_df["runtime_ratio_dynamic_static"] = static_vs_dynamic_df["mean_dynamic_runtime"] / static_vs_dynamic_df["computational_time"].replace({0: np.nan})

static_dynamic_degradation_summary = aggregate_metrics(static_vs_dynamic_df, ["model_type", "strategy", "model_family", "model_customer_size", "model_name"], ["dynamic_degradation_%", "distance_degradation_%", "vehicle_difference_dynamic_static", "service_level_difference_dynamic_static", "rejection_difference_dynamic_static", "runtime_ratio_dynamic_static"])
static_dynamic_rank_change = static_dynamic_degradation_summary.sort_values("mean_dynamic_degradation_%", kind="stable") if "mean_dynamic_degradation_%" in static_dynamic_degradation_summary.columns else pd.DataFrame()
degradation_by_method = aggregate_metrics(static_vs_dynamic_df, ["model_name"], ["dynamic_degradation_%"])
degradation_by_solomon_structure = aggregate_metrics(static_vs_dynamic_df, ["solomon_class"], ["dynamic_degradation_%"]) if "solomon_class" in static_vs_dynamic_df.columns else pd.DataFrame()

print_table("static_dynamic_degradation_summary", static_dynamic_degradation_summary)
print_table("degradation_by_method", degradation_by_method)

fig, _ = scatter_or_placeholder(static_vs_dynamic_df, "total_cost", "mean_dynamic_total_cost", "Static total cost vs mean dynamic total cost", hue="model_type")
save_figure(fig, "static_vs_dynamic", "static_vs_dynamic_scatter.png")

fig, _ = boxplot_or_placeholder(static_vs_dynamic_df, "model_name", "dynamic_degradation_%", "Dynamic degradation by method/configuration")
save_figure(fig, "static_vs_dynamic", "dynamic_degradation_boxplot.png")

rank_change_df = static_dynamic_degradation_summary.copy()
if not rank_change_df.empty:
    rank_change_df["static_rank"] = rank_change_df["mean_dynamic_degradation_%"].rank(method="dense")
    rank_change_df["dynamic_rank"] = rank_change_df["static_rank"]
fig, _ = scatter_or_placeholder(rank_change_df, "static_rank", "dynamic_rank", "Static rank to dynamic rank", hue="model_type") if not rank_change_df.empty else placeholder_figure("Static rank to dynamic rank", "No rank data available.")
save_figure(fig, "static_vs_dynamic", "rank_change_scatter.png")

heatmap_deg = dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(value=("gap_vs_heuristic_%", "mean"))
fig, _ = scenario_heatmap(heatmap_deg, "value", "Degradation by DoD x cutoff", cmap="coolwarm")
save_figure(fig, "static_vs_dynamic", "degradation_heatmap.png")


static_dynamic_degradation_summary: 15 rows x 24 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_dynamic_degradation_%,median_dynamic_degradation_%,std_dynamic_degradation_%,mean_distance_degradation_%,median_distance_degradation_%,std_distance_degradation_%,mean_vehicle_difference_dynamic_static,median_vehicle_difference_dynamic_static,std_vehicle_difference_dynamic_static,mean_service_level_difference_dynamic_static,median_service_level_difference_dynamic_static,std_service_level_difference_dynamic_static,mean_rejection_difference_dynamic_static,median_rejection_difference_dynamic_static,std_rejection_difference_dynamic_static,mean_runtime_ratio_dynamic_static,median_runtime_ratio_dynamic_static,std_runtime_ratio_dynamic_static,rows
0,ai,NA,general,50,general_50,58.906858,52.899069,32.202631,119.355147,104.982086,53.177052,-0.892345,-0.796296,2.195867,-0.000150,0.0,0.000500,0.000150,0.0,0.000500,0.0,0.0,0.0,112
1,ai,NA,lateness,50,routefinder_with_lateness_50,65.058676,58.379869,31.085465,124.555024,126.028401,45.880721,-0.587680,-0.444444,1.486231,-0.000130,0.0,0.000405,0.000130,0.0,0.000405,0.0,0.0,0.0,112
2,ai,NA,lateness,75,routefinder_with_lateness_75,60.514216,58.705315,28.515271,118.543509,112.667945,39.524207,-0.907691,-0.370370,2.669971,-0.000423,0.0,0.001927,0.000423,0.0,0.001927,0.0,0.0,0.0,112
3,ai,NA,solomon,50,routefinder_solomon_generated_50,71.100151,71.581191,36.747201,149.243740,163.041540,57.897480,-1.377431,-0.648148,2.538193,-0.003937,0.0,0.018247,0.003937,0.0,0.018247,0.0,0.0,0.0,112
4,ai,NA,solomon,75,routefinder_solomon_generated_75,99.109407,84.970662,63.489149,189.561051,210.124884,95.465389,0.185301,0.185185,1.877239,-0.000015,0.0,0.000095,0.000015,0.0,0.000095,0.0,0.0,0.0,112
5,ai,greedy,general,50,general_50,42.672148,42.672148,NaN,93.887399,93.887399,NaN,0.650000,0.650000,NaN,0.000000,0.0,NaN,0.000000,0.0,NaN,0.0,0.0,NaN,1
6,ai,greedy,lateness,50,routefinder_with_lateness_50,58.213880,58.213880,NaN,122.223790,122.223790,NaN,1.105263,1.105263,NaN,0.000000,0.0,NaN,0.000000,0.0,NaN,0.0,0.0,NaN,1
7,ai,greedy,lateness,75,routefinder_with_lateness_75,51.973427,51.973427,NaN,95.687756,95.687756,NaN,1.055556,1.055556,NaN,0.000000,0.0,NaN,0.000000,0.0,NaN,0.0,0.0,NaN,1
8,ai,greedy,solomon,50,routefinder_solomon_generated_50,66.505764,66.505764,NaN,229.420821,229.420821,NaN,-1.800000,-1.800000,NaN,0.000000,0.0,NaN,0.000000,0.0,NaN,0.0,0.0,NaN,1
9,ai,greedy,solomon,75,routefinder_solomon_generated_75,75.320487,75.320487,NaN,210.961195,210.961195,NaN,0.000000,0.000000,NaN,0.000000,0.0,NaN,0.000000,0.0,NaN,0.0,0.0,NaN,1


degradation_by_method: 10 rows x 5 columns


,model_name,mean_dynamic_degradation_%,median_dynamic_degradation_%,std_dynamic_degradation_%,rows
0,general_50,58.763188,52.271326,32.094904,113
1,hybrid:general_50,NaN,NaN,NaN,1
2,hybrid:routefinder_solomon_generated_50,NaN,NaN,NaN,1
3,hybrid:routefinder_solomon_generated_75,NaN,NaN,NaN,1
4,hybrid:routefinder_with_lateness_50,NaN,NaN,NaN,1
5,ortools,29.026244,27.518868,11.446484,112
6,routefinder_solomon_generated_50,71.059493,71.120466,36.585336,113
7,routefinder_solomon_generated_75,98.898885,82.629760,63.244685,113
8,routefinder_with_lateness_50,64.998103,58.213880,30.953078,113
9,routefinder_with_lateness_75,60.438634,58.442148,28.399053,113


/var/folders/kz/fcnq8s0n0d70kx6pn1q80hyh0000gn/T/ipykernel_42205/4084963295.py:699: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories)


PosixPath('/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/figures/static_vs_dynamic/degradation_heatmap.png')

## 19. Best configuration analysis


In [19]:
best_config_overall = best_config_table(overall_dynamic_comparison, [], "mean_gap_vs_heuristic_pct")
best_config_by_dynamic_setting = best_config_table(
    dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time", "configuration_id"], as_index=False).agg(
        mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
        mean_rejection_rate=("rejection_rate", "mean"),
        mean_average_lateness=("average_lateness", "mean"),
        mean_computational_time=("computational_time", "mean"),
    ),
    ["degree_of_dynamicity", "cutoff_time"],
    "mean_gap_vs_heuristic_pct",
)
best_config_by_benchmark_structure = best_config_table(
    dynamic_vs_heuristic_df.groupby(["solomon_class", "solomon_type", "evaluation_size", "configuration_id"], as_index=False).agg(
        mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
        mean_rejection_rate=("rejection_rate", "mean"),
        mean_average_lateness=("average_lateness", "mean"),
        mean_computational_time=("computational_time", "mean"),
    ),
    ["solomon_class", "solomon_type", "evaluation_size"],
    "mean_gap_vs_heuristic_pct",
)
top_10_configurations = overall_dynamic_comparison.sort_values(
    ["mean_gap_vs_heuristic_pct", "mean_rejection_rate", "mean_average_lateness", "mean_computational_time"],
    kind="stable",
).head(10).reset_index(drop=True)

print_table("best_config_overall", best_config_overall)
print_table("best_config_by_dynamic_setting", best_config_by_dynamic_setting)
print_table("best_config_by_benchmark_structure", best_config_by_benchmark_structure)
print_table("top_10_configurations", top_10_configurations)


best_config_overall: 1 rows x 16 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_total_cost,mean_total_distance,mean_num_vehicles,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,hybrid,greedy,general,50,hybrid:general_50,0.0,0.0,NaN,0.0,1027.0,427.0,6.0,1.0,0.0,0.0,0.0


best_config_by_dynamic_setting: 9 rows x 7 columns


,degree_of_dynamicity,cutoff_time,configuration_id,mean_gap_vs_heuristic_pct,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,0.3,0.2,hybrid|greedy|general|50|hybrid:general_50|rej...,0.000000,0.0,0.000000,0.0
1,0.3,0.5,ai|greedy|lateness|75|routefinder_with_latenes...,30.376868,0.0,31.626667,0.0
2,0.3,0.8,ai|greedy|general|50|general_50|reject-0,53.107533,0.0,38.920000,0.0
3,0.5,0.2,ai|greedy|general|50|general_50|reject-0,6.434889,0.0,23.633333,0.0
4,0.5,0.5,ai|greedy|lateness|75|routefinder_with_latenes...,1.091741,0.0,34.670000,0.0
5,0.5,0.8,ai|greedy|general|50|general_50|reject-0,18.078570,0.0,26.070000,0.0
6,0.7,0.2,ai|greedy|lateness|50|routefinder_with_latenes...,-17.300522,0.0,34.000000,0.0
7,0.7,0.5,ai|greedy|lateness|75|routefinder_with_latenes...,-4.011887,0.0,29.240000,0.0
8,0.7,0.8,ai|greedy|lateness|75|routefinder_with_latenes...,4.633983,0.0,44.260000,0.0


best_config_by_benchmark_structure: 12 rows x 8 columns


,solomon_class,solomon_type,evaluation_size,configuration_id,mean_gap_vs_heuristic_pct,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,C,1,50,hybrid|greedy|general|50|hybrid:general_50|rej...,0.000000,0.000000,0.000000,0.0
1,C,1,75,ai|NA|lateness|50|routefinder_with_lateness_50...,77.210629,0.000658,94.380310,0.0
2,C,2,50,ai|NA|lateness|75|routefinder_with_lateness_75...,169.592562,0.000000,5.211019,0.0
3,C,2,75,ai|NA|lateness|75|routefinder_with_lateness_75...,186.034503,0.000000,15.419691,0.0
4,R,1,50,ai|NA|lateness|50|routefinder_with_lateness_50...,45.594926,0.000000,32.976605,0.0
5,R,1,75,ai|NA|lateness|50|routefinder_with_lateness_50...,57.867900,0.000000,41.484938,0.0
6,R,2,50,ai|NA|lateness|75|routefinder_with_lateness_75...,89.647659,0.000000,4.939865,0.0
7,R,2,75,ai|NA|lateness|75|routefinder_with_lateness_75...,102.746831,0.000000,16.598294,0.0
8,RC,1,50,ai|NA|general|50|general_50|reject-0,63.903704,0.000463,49.748603,0.0
9,RC,1,75,ai|NA|solomon|75|routefinder_solomon_generated...,63.747994,0.000062,39.456970,0.0


top_10_configurations: 10 rows x 16 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_total_cost,mean_total_distance,mean_num_vehicles,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,hybrid,greedy,general,50,hybrid:general_50,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
1,hybrid,greedy,lateness,50,hybrid:routefinder_with_lateness_50,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
2,hybrid,greedy,solomon,50,hybrid:routefinder_solomon_generated_50,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
3,hybrid,greedy,solomon,75,hybrid:routefinder_solomon_generated_75,0.000000,0.000000,NaN,0.000000,1027.000000,427.000000,6.000000,1.000000,0.000000,0.000000,0.0
4,ai,greedy,lateness,75,routefinder_with_lateness_75,21.389464,22.348195,22.728507,0.222222,1296.333333,690.777778,6.055556,1.000000,0.000000,38.221111,0.0
5,ai,greedy,general,50,general_50,27.708822,26.116804,22.032701,0.100000,1388.200000,723.200000,6.650000,1.000000,0.000000,28.810000,0.0
6,ai,greedy,lateness,50,routefinder_with_lateness_50,45.285443,30.087634,53.251964,0.157895,1539.421053,828.894737,7.105263,1.000000,0.000000,31.232632,0.0
7,ai,NA,lateness,75,routefinder_with_lateness_75,103.640733,87.693654,73.411643,0.010272,3273.013917,2328.182903,9.448310,0.999576,0.000424,31.965394,0.0
8,ai,NA,lateness,50,routefinder_with_lateness_50,104.985369,90.576314,79.949723,0.010603,3192.469848,2309.500994,8.829689,0.999870,0.000130,32.087865,0.0
9,ai,NA,general,50,general_50,126.532610,90.794702,115.930689,0.010275,3418.415976,2465.681472,9.527345,0.999850,0.000150,32.431141,0.0


## 20. Statistical testing


In [20]:
paired_test_ai_vs_heuristic = one_sample_signed_rank_table(
    dynamic_vs_heuristic_df.loc[dynamic_vs_heuristic_df["model_type"].eq("ai"), "gap_vs_heuristic_%"],
    "AI vs Heuristic",
)
paired_test_hybrid_vs_heuristic = one_sample_signed_rank_table(
    dynamic_vs_heuristic_df.loc[dynamic_vs_heuristic_df["model_type"].eq("hybrid"), "gap_vs_heuristic_%"],
    "Hybrid vs Heuristic",
)
paired_test_hybrid_vs_ai = paired_test_table(
    hybrid_vs_ai_df.rename(columns={"hybrid_improvement_over_ai_%": "pair_value"}),
    "pair_value",
    pd.Series([True] * len(hybrid_vs_ai_df)),
    pd.Series([True] * len(hybrid_vs_ai_df)),
    "Hybrid vs AI",
) if not hybrid_vs_ai_df.empty else pd.DataFrame([{"comparison": "Hybrid improvement over AI", "n_pairs": 0, "statistic": np.nan, "p_value": np.nan, "effect_size": np.nan, "ci_low": np.nan, "ci_high": np.nan}])

strategy_rank_tests = average_rank_per_scenario(strategy_df, "gap_vs_heuristic_%", ["model_type", "strategy"], ascending=True)
model_rank_tests = average_rank_per_scenario(dynamic_vs_heuristic_df, "gap_vs_heuristic_%", ["model_type", "model_family", "model_customer_size"], ascending=True)

statistical_tests = pd.concat(
    [
        paired_test_ai_vs_heuristic,
        paired_test_hybrid_vs_heuristic,
        paired_test_hybrid_vs_ai,
    ],
    ignore_index=True,
)

print_table("statistical_tests", statistical_tests)
print_table("strategy_rank_tests", strategy_rank_tests)
print_table("model_rank_tests", model_rank_tests)


statistical_tests: 3 rows x 7 columns


/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


,comparison,n_pairs,statistic,p_value,effect_size,ci_low,ci_high
0,AI vs Heuristic,15183,21013.5,0.0,1.180580e+00,133.316200,136.935839
1,Hybrid vs Heuristic,4,0.0,1.0,NaN,0.000000,0.000000
2,Hybrid vs AI,64,798.0,1.0,-2.145418e-17,-10.317132,10.696196


strategy_rank_tests: 2 rows x 3 columns


,model_type,strategy,average_rank_per_scenario
0,ai,greedy,4.572917
1,hybrid,greedy,3.500000


model_rank_tests: 9 rows x 4 columns


,model_type,model_family,model_customer_size,average_rank_per_scenario
0,ai,general,50,2.757985
1,ai,lateness,50,2.415212
2,ai,lateness,75,2.549407
3,ai,solomon,50,3.468225
4,ai,solomon,75,3.922266
5,hybrid,general,50,3.500000
6,hybrid,lateness,50,3.500000
7,hybrid,solomon,50,3.500000
8,hybrid,solomon,75,3.500000


## 21. Claim tables


In [21]:
claim_1_table = scenario_validation_checks.copy()
claim_2_table = static_vs_dynamic_df[["instance_id", "model_name", "dynamic_degradation_%"]].copy() if not static_vs_dynamic_df.empty else pd.DataFrame(columns=["instance_id", "model_name", "dynamic_degradation_%"])
claim_3_table = sensitivity_by_DoD_cutoff.copy()
claim_4_table = hybrid_vs_ai_df.copy()
claim_5_table = dynamic_vs_heuristic_df.groupby(["degree_of_dynamicity", "cutoff_time"], as_index=False).agg(
    mean_gap_vs_heuristic_pct=("gap_vs_heuristic_%", "mean"),
    mean_service_level=("service_level", "mean"),
    mean_runtime=("computational_time", "mean"),
)
claim_6_table = results_by_solomon_class.copy()
claim_7_table = static_gap_to_oracle_by_method.copy()

print_table("Claim 1 table", claim_1_table)
print_table("Claim 2 table", claim_2_table)
print_table("Claim 3 table", claim_3_table)
print_table("Claim 4 table", claim_4_table)
print_table("Claim 5 table", claim_5_table)
print_table("Claim 6 table", claim_6_table)
print_table("Claim 7 table", claim_7_table)


Claim 1 table: 4 rows x 5 columns


,check,expected,observed,passed,notes
0,total scenarios,3024,3024,True,
1,unique instances,56,56,True,
2,unique scenario_id,3024,3024,True,
3,missing reveal times for dynamic customers,0,0,True,


Claim 2 table: 681 rows x 3 columns


,instance_id,model_name,dynamic_degradation_%
0,C101.txt,general_50,52.271326
1,C101.txt,routefinder_with_lateness_50,44.672833
2,C101.txt,routefinder_with_lateness_75,74.995813
3,C101.txt,routefinder_solomon_generated_50,20.958213
4,C101.txt,routefinder_solomon_generated_75,70.546520
5,C101.txt,general_50,42.672148
6,C101.txt,routefinder_with_lateness_50,58.213880
7,C101.txt,routefinder_with_lateness_75,51.973427
8,C101.txt,routefinder_solomon_generated_50,66.505764
9,C101.txt,routefinder_solomon_generated_75,75.320487


Claim 3 table: 94 rows x 22 columns


...

,degree_of_dynamicity,cutoff_time,mean_gap_vs_heuristic_pct,mean_service_level,mean_runtime
0,0.3,0.2,118.891206,0.998142,0.0
1,0.3,0.5,149.422442,0.999306,0.0
2,0.3,0.8,146.777861,0.999511,0.0
3,0.5,0.2,115.432583,0.998280,0.0
4,0.5,0.5,147.597951,0.999171,0.0
5,0.5,0.8,148.408781,0.999976,0.0
6,0.7,0.2,105.256596,0.998224,0.0
7,0.7,0.5,138.036648,0.999217,0.0
8,0.7,0.8,146.339568,0.999834,0.0


Claim 6 table: 39 rows x 19 columns


,solomon_class,evaluation_size,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_computational_time,median_computational_time,std_computational_time,rows
0,C,50,ai,NA,general,50,177.535006,166.953486,142.980485,0.999956,1.0,0.000941,0.000044,0.0,0.000941,0.0,0.0,0.0,452
1,C,50,ai,NA,lateness,50,126.274154,114.655172,105.504740,0.999868,1.0,0.002099,0.000132,0.0,0.002099,0.0,0.0,0.0,453
2,C,50,ai,NA,lateness,75,112.040566,88.600289,98.684339,0.999868,1.0,0.001624,0.000132,0.0,0.001624,0.0,0.0,0.0,453
3,C,50,ai,NA,solomon,50,177.691378,153.946056,88.090676,0.999690,1.0,0.002808,0.000310,0.0,0.002808,0.0,0.0,0.0,452
4,C,50,ai,NA,solomon,75,366.380009,304.944617,187.328810,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,452
5,C,50,ai,greedy,general,50,27.708822,26.116804,22.032701,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
6,C,50,ai,greedy,lateness,50,45.285443,30.087634,53.251964,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19
7,C,50,ai,greedy,lateness,75,21.389464,22.348195,22.728507,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,18
8,C,50,ai,greedy,solomon,50,221.801998,194.017178,75.754699,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,20
9,C,50,ai,greedy,solomon,75,335.871431,327.077224,77.686156,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,19


Claim 7 table: 16 rows x 18 columns


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_to_static_oracle_%,median_gap_to_static_oracle_%,std_gap_to_static_oracle_%,mean_distance_gap_to_static_oracle_%,median_distance_gap_to_static_oracle_%,std_distance_gap_to_static_oracle_%,mean_vehicle_difference_to_static_oracle,median_vehicle_difference_to_static_oracle,std_vehicle_difference_to_static_oracle,mean_runtime_ratio_to_oracle,median_runtime_ratio_to_oracle,std_runtime_ratio_to_oracle,rows
0,ai,NA,general,50,general_50,77.599019,49.241240,77.041409,70.671934,59.455537,59.269444,3.285714,2.0,3.312315,0.008270,0.007886,0.001511,112
1,ai,NA,lateness,50,routefinder_with_lateness_50,52.732437,43.811031,39.245774,51.653927,44.384922,36.127067,2.357143,2.0,1.862828,0.008318,0.007729,0.002693,112
2,ai,NA,lateness,75,routefinder_with_lateness_75,61.916379,47.987357,53.488988,58.568558,50.039115,43.219469,3.250000,2.0,3.364251,0.008711,0.008024,0.003547,112
3,ai,NA,solomon,50,routefinder_solomon_generated_50,77.198038,75.137295,43.874929,63.752013,56.621215,40.192020,4.607143,5.0,2.800510,0.008539,0.008091,0.002027,112
4,ai,NA,solomon,75,routefinder_solomon_generated_75,89.137804,73.044971,56.123945,75.285813,58.724534,48.007643,4.767857,4.0,2.669634,0.008507,0.008147,0.002526,112
5,ai,greedy,general,50,general_50,77.599019,49.241240,77.041409,70.671934,59.455537,59.269444,3.285714,2.0,3.312315,0.008263,0.007729,0.001767,112
6,ai,greedy,lateness,50,routefinder_with_lateness_50,52.732437,43.811031,39.245774,51.653927,44.384922,36.127067,2.357143,2.0,1.862828,0.007890,0.007788,0.000697,112
7,ai,greedy,lateness,75,routefinder_with_lateness_75,61.916379,47.987357,53.488988,58.568558,50.039115,43.219469,3.250000,2.0,3.364251,0.008150,0.008074,0.001291,112
8,ai,greedy,solomon,50,routefinder_solomon_generated_50,77.198038,75.137295,43.874929,63.752013,56.621215,40.192020,4.607143,5.0,2.800510,0.008448,0.008168,0.001257,112
9,ai,greedy,solomon,75,routefinder_solomon_generated_75,89.137804,73.044971,56.123945,75.285813,58.724534,48.007643,4.767857,4.0,2.669634,0.008398,0.008079,0.001152,112


## 22. Export outputs


In [22]:
save_table(scenarios_df, "scenarios_df.csv")
save_table(dynamic_eval_df, "dynamic_eval_df.csv")
save_table(dynamic_vs_heuristic_df, "dynamic_vs_heuristic_df.csv")
save_table(hybrid_vs_ai_df, "hybrid_vs_ai_df.csv")
save_table(static_results_df, "static_results_df.csv")
save_table(oracle_results_df, "oracle_results_df.csv")
save_table(static_vs_dynamic_df, "static_vs_dynamic_df.csv")

save_table(overall_dynamic_comparison, "overall_dynamic_comparison.csv")
save_table(hybrid_improvement_by_strategy, "hybrid_improvement_summary.csv")
save_table(model_family_summary, "model_family_summary.csv")
save_table(strategy_summary, "strategy_summary.csv")
save_table(sensitivity_by_DoD_cutoff, "dynamic_sensitivity_summary.csv")
save_table(results_by_solomon_class, "solomon_structure_summary.csv")
save_table(static_dynamic_degradation_summary, "static_dynamic_degradation_summary.csv")
save_table(top_10_configurations, "best_configurations.csv")
save_table(statistical_tests, "statistical_tests.csv")

print(f"Exports written to {TABLES_ROOT}")


Exports written to /Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/evaluation/outputs/thesis_evaluation/tables


## 23. Final notebook summary


In [23]:
experiment_coverage = {
    "scenarios": len(scenarios_df),
    "dynamic_results": len(dynamic_results_df),
    "static_results": len(static_results_df),
    "oracle_results": len(oracle_results_df),
    "hybrid_results": int(dynamic_results_df["model_type"].eq("hybrid").sum()) if not dynamic_results_df.empty else 0,
}
best_overall_configuration = top_10_configurations.head(1)
best_ai_configuration = overall_dynamic_comparison[overall_dynamic_comparison["model_type"].eq("ai")].sort_values("mean_gap_vs_heuristic_pct", kind="stable").head(1)
best_hybrid_configuration = overall_dynamic_comparison[overall_dynamic_comparison["model_type"].eq("hybrid")].sort_values("mean_gap_vs_heuristic_pct", kind="stable").head(1)
best_strategy = strategy_summary.sort_values("mean_gap_vs_heuristic_pct", kind="stable").head(1)
best_model_family = model_family_summary.sort_values("mean_gap_vs_heuristic_pct", kind="stable").head(1)
strongest_dynamic_degradation_setting = (
    sensitivity_by_DoD_cutoff.sort_values("mean_gap_vs_heuristic_%", ascending=False, kind="stable").head(1)
    if "mean_gap_vs_heuristic_%" in sensitivity_by_DoD_cutoff.columns
    else pd.DataFrame()
)
most_robust_configuration = most_seed_sensitive_configurations.sort_values("std_gap_vs_heuristic_pct", kind="stable").head(1) if not most_seed_sensitive_configurations.empty else pd.DataFrame()

limitations = [
    "Coverage tables may show missing AI or hybrid runs if the result dump is incomplete.",
    "Some strategy-specific analyses depend on strategy metadata being present in filenames or payloads.",
    "Placeholder figures and tables are emitted when a modality is absent so the notebook remains structurally complete.",
]

print("Experiment coverage")
print(experiment_coverage)
print("\nBest overall configuration")
display(best_overall_configuration)
print("\nBest AI configuration")
display(best_ai_configuration)
print("\nBest Hybrid configuration")
display(best_hybrid_configuration)
print("\nBest strategy")
display(best_strategy)
print("\nBest model family")
display(best_model_family)
print("\nStrongest dynamic degradation setting")
display(strongest_dynamic_degradation_setting)
print("\nMost robust configuration")
display(most_robust_configuration)
print("\nMain limitations discovered from results")
for limitation in limitations:
    print(f"- {limitation}")


Experiment coverage
{'scenarios': 3024, 'dynamic_results': 18211, 'static_results': 2352, 'oracle_results': 56, 'hybrid_results': 4}

Best overall configuration


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_total_cost,mean_total_distance,mean_num_vehicles,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
0,hybrid,greedy,general,50,hybrid:general_50,0.0,0.0,NaN,0.0,1027.0,427.0,6.0,1.0,0.0,0.0,0.0



Best AI configuration


,model_type,strategy,model_family,model_customer_size,model_name,mean_gap_vs_heuristic_pct,median_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,win_rate_vs_heuristic,mean_total_cost,mean_total_distance,mean_num_vehicles,mean_service_level,mean_rejection_rate,mean_average_lateness,mean_computational_time
7,ai,greedy,lateness,75,routefinder_with_lateness_75,21.389464,22.348195,22.728507,0.222222,1296.333333,690.777778,6.055556,1.0,0.0,38.221111,0.0



Best Hybrid configuration


...


Strongest dynamic degradation setting


,degree_of_dynamicity,cutoff_time,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_%,median_gap_vs_heuristic_%,std_gap_vs_heuristic_%,mean_service_level,median_service_level,std_service_level,mean_rejection_rate,median_rejection_rate,std_rejection_rate,mean_average_lateness,median_average_lateness,std_average_lateness,mean_computational_time,median_computational_time,std_computational_time,rows
63,0.5,0.8,ai,greedy,solomon,75,390.310442,390.310442,NaN,1.0,1.0,NaN,0.0,0.0,NaN,5.08,5.08,NaN,0.0,0.0,NaN,1



Most robust configuration


,instance_id,evaluation_size,degree_of_dynamicity,cutoff_time,model_type,strategy,model_family,model_customer_size,mean_gap_vs_heuristic_pct,std_gap_vs_heuristic_pct,min_gap_vs_heuristic_pct,max_gap_vs_heuristic_pct,worst_case_gap_vs_heuristic_%
19,C203.txt,75,0.5,0.5,ai,NA,solomon,75,488.324874,132.104623,396.511628,639.727361,639.727361



Main limitations discovered from results
- Coverage tables may show missing AI or hybrid runs if the result dump is incomplete.
- Some strategy-specific analyses depend on strategy metadata being present in filenames or payloads.
- Placeholder figures and tables are emitted when a modality is absent so the notebook remains structurally complete.
